# 0. NOTEBOOK HEADER & METADATA

## Notebook Metadata

| Field | Value |
|---|---|
| **Target Table** | `brk.fact_profit` |
| **Product Group and Division** | To Be Updated By DJJ team |
| **Author** | Shivani Bahuguna |
| **Created** | 2026-04-09 |
| **DevOps ID and Link** | To Be Updated By DJJ team |
| **Load Type** | `Incremental` |
| **Grain** | To Be Updated By DJJ team |

---

## Description

> ETL to load Brk.factProfit from Dynamics and D365 financial data

%md
## Change Log

| Date | Author | DevOps ID and Link | Change Description |
|---|---|---|---|
| 2026-04-09 | Shivani Bahuguna | NA | Updates made to the notebook as per the standardization template |

# 1. CONFIGURATION, IMPORTS, & PARAMETERS

In [0]:
# ========================================
# LIBRARY IMPORTS
# ========================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
from delta.tables import DeltaTable
import uuid

spark.conf.set("spark.sql.session.timeZone", "America/New_York")

In [0]:
# ==============================================
# CONFIGURATION
# ==============================================

bronze_catalog = "bronze_non_prod"                                    #Catalog name for source table
SourceSchemaName = "djj"                                              #Schema name for source table
SourceTableName = "d365fo_ods_d365fo_brkgeneraljournalaccountentry"    #Table name for source table
enriched_catalog = "djj_enriched_non_prod"                            #Catalog name for target table
DestinationSchemaName = "brk"                                         #Schema name for target table
DestinationTableName = "fact_profit"                                   #Table name for target table
etl_runtime = datetime.now()                                          #Current Runtime for updating LastUpdateTime
temp_schema = "temp"                                                  #Schema name for temp table
MonthToLoad = 888888
layer = "enriched"                                                                    #Target layer
source_table = f"{bronze_catalog}.{SourceSchemaName}.{SourceTableName}"
target_table = f"{enriched_catalog}.{DestinationSchemaName}.{DestinationTableName}"
proc = "incremental_merge"                                                             #Load type
platform = "mssql"                                                                     #Source platform

# 2. START METADATA LOGGING

In [0]:
%run ../Metadata_Logger/execution_logger

Warning you are using the ipython `%run` line magic. To use the databricks `%run` cell magic make sure that the magic is at the very start of the cell.

In [0]:
# ===================================
# START METADATA LOGGING
# ===================================

run_ctx = start_execution_run(
        layer=layer,
        source_table=source_table,
        target_table=target_table,
        proc=proc,
        platform=platform
)

# 3. ETL LOGIC

In [0]:
# ========================================
# DECLARING VARIABLE : CurrentFiscalMonth
# ========================================

CurrentFiscalMonth = spark.sql(f"""
    SELECT MIN(Fiscal_Month) AS fm FROM {enriched_catalog}.{SourceSchemaName}.dim_date
    WHERE CAST(Date_Day AS DATE) = CURRENT_DATE()
""").first()["fm"]



# ========================================
# DECLARING VARIABLE : PriorFiscalMonth
# ========================================

PriorFiscalMonth = spark.sql(f"""
    SELECT MIN(Fiscal_Month) AS fm FROM {enriched_catalog}.{SourceSchemaName}.dim_date
    WHERE CAST(Date_Day AS DATE) = (
        SELECT DATE_SUB(MIN(Date_Day), 1) 
        FROM {enriched_catalog}.{SourceSchemaName}.dim_date 
        WHERE Fiscal_Month = {CurrentFiscalMonth}
    )
""").first()["fm"]



# ========================================
# DECLARING VARIABLE : SourceSystemKey
# ========================================

SourceSystemKey = spark.sql(f"""
SELECT MIN(SourceSystemKey) AS k
FROM {enriched_catalog}.{SourceSchemaName}.dim_sourcesystem
WHERE SystemName = 'MS Dynamics 365'
""").first()["k"]



# ========================================
# DECLARING VARIABLE : BrokerageGroupKey
# ========================================

BrokerageGroupKey = spark.sql(f"""
SELECT MIN(BrokerageGroupKey) AS k
FROM {enriched_catalog}.{DestinationSchemaName}.dim_brokeragegroup
WHERE BrokerageGroup = 'Ferrous Brokerage'
""").first()["k"]



# ========================================
# DECLARING VARIABLE : DefaultOfficeKey
# ========================================

DefaultOfficeKey = spark.sql(f"""
SELECT MIN(BrokerageOfficeKey) AS k
FROM {enriched_catalog}.{DestinationSchemaName}.dim_brokerageoffice
WHERE BrokerageGroup = 'Ferrous Brokerage'
  AND Office = 'Unknown'
""").first()["k"]



# ========================================
# DECLARING VARIABLE : ProfitPropertiesKey
# ========================================

ProfitPropertiesKey = spark.sql(f"""
SELECT MIN(ProfitPropertiesKey) AS k
FROM {enriched_catalog}.{DestinationSchemaName}.dim_profitproperties
WHERE ConsumerName = 'See Consumer Dimension'
  AND SupplierName = 'See Supplier Dimension'
  AND PurchaseGradeDescription = 'See Purchased Grade Dimension'
  AND SaleGradeDescription = 'See Sale Grade Dimension'
""").first()["k"]



# ========================================
# DECLARING VARIABLE : DefaultShipmentPropertiesKey
# ========================================

DefaultShipmentPropertiesKey = spark.sql(f"""
SELECT MIN(ShipmentPropertiesKey) AS k
FROM {enriched_catalog}.{DestinationSchemaName}.dim_shipmentproperties
WHERE ContractTypeCode = 'U'
  AND MultiLegCode = -1
  AND FreightResponsibilityCode = 'NOF'
  AND PaymentStatusCode = 'U'
  AND ShipmentStatusCode = -1
  AND Rejected = 'Not Rejected'
""").first()["k"]

In [0]:
# ========================================
# CREATING TEMP TABLE : temp_brk_fiscalmonthdates_d365
# ========================================

spark.sql(f"""CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_fiscalmonthdates_d365 AS
    SELECT Fiscal_Month, MAX(DateKey) AS DateKey FROM {enriched_catalog}.{SourceSchemaName}.dim_date GROUP BY Fiscal_Month""")



# ========================================
# CREATING TEMP TABLE : temp_brk_grades_d365
# ========================================

spark.sql(f"""CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_grades_d365 AS
    SELECT GradeCode, MIN(GradeKey) AS GradeKey FROM {enriched_catalog}.{DestinationSchemaName}.dim_grade GROUP BY GradeCode""")



# ========================================
# CREATING TEMP TABLE : temp_brk_mgbushnell_d365
# ========================================

spark.sql(f"""CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_mgbushnell_d365 AS
    SELECT GL.GENERALJOURNALACCOUNTENTRYRECID
    FROM {bronze_catalog}.{SourceSchemaName}.{SourceTableName} GL
    WHERE GL.IsDeleted = 0 
      AND GL.LEDGERNAME = 'BRK'
      AND GL.LEDGERACCOUNT LIKE '%-126000-%'""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# CREATING TEMP TABLE : temp_brk_glstaging_d365
# ========================================

spark.sql(f"""
          CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 (
    GENERALJOURNALACCOUNTENTRYRECID BIGINT,
    DataSource STRING COLLATE UTF8_LCASE,
    FiscalMonth INT,
    FiscalMonthDateKey INT,
    ACCOUNTINGDATE DATE,
    LEDGERACCOUNT STRING COLLATE UTF8_LCASE,
    VOUCHER STRING COLLATE UTF8_LCASE,
    DOCUMENTNUMBER STRING COLLATE UTF8_LCASE,
    GLAccountFirstDigit STRING COLLATE UTF8_LCASE,
    GLAccount STRING COLLATE UTF8_LCASE,
    BusinessUnit STRING COLLATE UTF8_LCASE,
    Office STRING COLLATE UTF8_LCASE,
    SaleType STRING COLLATE UTF8_LCASE,
    PurchaseShipmentNumber STRING COLLATE UTF8_LCASE,
    Description STRING COLLATE UTF8_LCASE,
    TRANSACTIONCURRENCYAMOUNT DECIMAL(32,6),
    TRANSACTIONCURRENCYDEBITAMOUNT DECIMAL(32,6),
    TRANSACTIONCURRENCYCREDITAMOUNT DECIMAL(32,6),
    POSTINGTYPE STRING COLLATE UTF8_LCASE,
    JOURNALCATEGORY STRING COLLATE UTF8_LCASE,
    Cost DECIMAL(32,6),
    SaleAmount DECIMAL(32,6)
)
""")



# ========================================
# INSERTING INTO TEMP TABLE : temp_brk_glstaging_d365
# ========================================

if MonthToLoad == 999999:
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
    (
        GENERALJOURNALACCOUNTENTRYRECID,
        FiscalMonth,
        ACCOUNTINGDATE,
        LEDGERACCOUNT,
        VOUCHER,
        DOCUMENTNUMBER,
        TRANSACTIONCURRENCYAMOUNT,
        TRANSACTIONCURRENCYDEBITAMOUNT,
        TRANSACTIONCURRENCYCREDITAMOUNT,
        POSTINGTYPE,
        JOURNALCATEGORY,
        Description
    )
    SELECT  
        GL.GENERALJOURNALACCOUNTENTRYRECID,
        TD.Fiscal_Month,
        CAST(GL.ACCOUNTINGDATE AS DATE),
        GL.LEDGERACCOUNT,
        GL.VOUCHER,
        GL.DOCUMENTNUMBER,
        GL.TRANSACTIONCURRENCYAMOUNT,
        GL.TRANSACTIONCURRENCYDEBITAMOUNT,
        GL.TRANSACTIONCURRENCYCREDITAMOUNT,
        GL.POSTINGTYPE,
        GL.JOURNALCATEGORY,
        GL.DESCRIPTION
    FROM {bronze_catalog}.{SourceSchemaName}.{SourceTableName} GL
    INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
        ON CAST(GL.ACCOUNTINGDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.LEDGERNAME = 'BRK'
      AND TD.Fiscal_Month = {CurrentFiscalMonth}
    """)

elif MonthToLoad == 888888:
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 (
        GENERALJOURNALACCOUNTENTRYRECID,
        FiscalMonth,
        ACCOUNTINGDATE,
        LEDGERACCOUNT,
        VOUCHER,
        DOCUMENTNUMBER,
        TRANSACTIONCURRENCYAMOUNT,
        TRANSACTIONCURRENCYDEBITAMOUNT,
        TRANSACTIONCURRENCYCREDITAMOUNT,
        POSTINGTYPE,
        JOURNALCATEGORY,
        Description
    )
    SELECT  
        GL.GENERALJOURNALACCOUNTENTRYRECID,
        TD.Fiscal_Month,
        CAST(GL.ACCOUNTINGDATE AS DATE),
        GL.LEDGERACCOUNT,
        GL.VOUCHER,
        GL.DOCUMENTNUMBER,
        GL.TRANSACTIONCURRENCYAMOUNT,
        GL.TRANSACTIONCURRENCYDEBITAMOUNT,
        GL.TRANSACTIONCURRENCYCREDITAMOUNT,
        GL.POSTINGTYPE,
        GL.JOURNALCATEGORY,
        GL.DESCRIPTION
    FROM {bronze_catalog}.{SourceSchemaName}.{SourceTableName} GL
    INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
        ON CAST(GL.ACCOUNTINGDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.LEDGERNAME = 'BRK'
      AND TD.Fiscal_Month IN ({CurrentFiscalMonth}, {PriorFiscalMonth})
    """)

elif MonthToLoad == 777777:
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 (
        GENERALJOURNALACCOUNTENTRYRECID,
        FiscalMonth,
        ACCOUNTINGDATE,
        LEDGERACCOUNT,
        VOUCHER,
        DOCUMENTNUMBER,
        TRANSACTIONCURRENCYAMOUNT,
        TRANSACTIONCURRENCYDEBITAMOUNT,
        TRANSACTIONCURRENCYCREDITAMOUNT,
        POSTINGTYPE,
        JOURNALCATEGORY,
        Description
    )
    SELECT  
        GL.GENERALJOURNALACCOUNTENTRYRECID,
        TD.Fiscal_Month,
        CAST(GL.ACCOUNTINGDATE AS DATE),
        GL.LEDGERACCOUNT,
        GL.VOUCHER,
        GL.DOCUMENTNUMBER,
        GL.TRANSACTIONCURRENCYAMOUNT,
        GL.TRANSACTIONCURRENCYDEBITAMOUNT,
        GL.TRANSACTIONCURRENCYCREDITAMOUNT,
        GL.POSTINGTYPE,
        GL.JOURNALCATEGORY,
        GL.DESCRIPTION
    FROM {bronze_catalog}.{SourceSchemaName}.{SourceTableName} GL
    INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
        ON CAST(GL.ACCOUNTINGDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.LEDGERNAME = 'BRK'
      AND TD.Fiscal_Month = {PriorFiscalMonth}
    """)

elif MonthToLoad == 111111:
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 (
        GENERALJOURNALACCOUNTENTRYRECID,
        FiscalMonth,
        ACCOUNTINGDATE,
        LEDGERACCOUNT,
        VOUCHER,
        DOCUMENTNUMBER,
        TRANSACTIONCURRENCYAMOUNT,
        TRANSACTIONCURRENCYDEBITAMOUNT,
        TRANSACTIONCURRENCYCREDITAMOUNT,
        POSTINGTYPE,
        JOURNALCATEGORY,
        Description
    )
    SELECT  
        GL.GENERALJOURNALACCOUNTENTRYRECID,
        TD.Fiscal_Month,
        CAST(GL.ACCOUNTINGDATE AS DATE),
        GL.LEDGERACCOUNT,
        GL.VOUCHER,
        GL.DOCUMENTNUMBER,
        GL.TRANSACTIONCURRENCYAMOUNT,
        GL.TRANSACTIONCURRENCYDEBITAMOUNT,
        GL.TRANSACTIONCURRENCYCREDITAMOUNT,
        GL.POSTINGTYPE,
        GL.JOURNALCATEGORY,
        GL.DESCRIPTION
    FROM {bronze_catalog}.{SourceSchemaName}.{SourceTableName} GL
    INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
        ON CAST(GL.ACCOUNTINGDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.LEDGERNAME = 'BRK'
    """)

else:
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 (
        GENERALJOURNALACCOUNTENTRYRECID,
        FiscalMonth,
        ACCOUNTINGDATE,
        LEDGERACCOUNT,
        VOUCHER,
        DOCUMENTNUMBER,
        TRANSACTIONCURRENCYAMOUNT,
        TRANSACTIONCURRENCYDEBITAMOUNT,
        TRANSACTIONCURRENCYCREDITAMOUNT,
        POSTINGTYPE,
        JOURNALCATEGORY,
        Description
    )
    SELECT  
        GL.GENERALJOURNALACCOUNTENTRYRECID,
        TD.Fiscal_Month,
        CAST(GL.ACCOUNTINGDATE AS DATE),
        GL.LEDGERACCOUNT,
        GL.VOUCHER,
        GL.DOCUMENTNUMBER,
        GL.TRANSACTIONCURRENCYAMOUNT,
        GL.TRANSACTIONCURRENCYDEBITAMOUNT,
        GL.TRANSACTIONCURRENCYCREDITAMOUNT,
        GL.POSTINGTYPE,
        GL.JOURNALCATEGORY,
        GL.DESCRIPTION
    FROM {bronze_catalog}.{SourceSchemaName}.{SourceTableName} GL
    INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
        ON CAST(GL.ACCOUNTINGDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.LEDGERNAME = 'BRK'
      AND TD.Fiscal_Month = {MonthToLoad}
    """)

In [0]:
# ========================================
# if the transaction occurred in the current fiscal month, then it is considered "Summary"; if it occurred in a prior fiscal month, then it is considered "History"
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
SET DataSource = CASE 
    WHEN FiscalMonth = {CurrentFiscalMonth} THEN 'Summary'
    ELSE 'History'
END
""")



# ========================================
# the last day of the fiscal month that the transaction occurred in
# ========================================

spark.sql(f"""MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 GL USING {enriched_catalog}.{temp_schema}.temp_brk_fiscalmonthdates_d365 TD2
    ON GL.FiscalMonth = TD2.Fiscal_Month WHEN MATCHED THEN UPDATE SET GL.FiscalMonthDateKey = TD2.DateKey""")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
    SET GLAccountFirstDigit = substring(split(cast(LEDGERACCOUNT as string), '-')[0], 1, 1)""")


spark.sql(f"DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 WHERE GLAccountFirstDigit NOT IN ('4','5')")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
    SET GLAccount = SPLIT(cast(LEDGERACCOUNT as string), '-')[0]""")


spark.sql(f"DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 WHERE GLAccount IN ('51120','51125','51130')")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
    SET BusinessUnit = SPLIT(cast(LEDGERACCOUNT as string), '-')[2]""")


spark.sql(f"DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 WHERE BusinessUnit = '147000'")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
    SET Office = SPLIT(cast(LEDGERACCOUNT as string), '-')[3]""")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
    SET SaleType = SPLIT(cast(LEDGERACCOUNT as string), '-')[5]""")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
    SET PurchaseShipmentNumber = SPLIT(cast(LEDGERACCOUNT as string), '-')[6]""")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 SET Cost = TRANSACTIONCURRENCYAMOUNT WHERE GLAccountFirstDigit = '5'""")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 SET SaleAmount = TRANSACTIONCURRENCYAMOUNT * -1 WHERE GLAccountFirstDigit = '4'""")


spark.sql(f"""DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
    WHERE GENERALJOURNALACCOUNTENTRYRECID IN (SELECT GENERALJOURNALACCOUNTENTRYRECID FROM {enriched_catalog}.{temp_schema}.temp_brk_mgbushnell_d365)""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# CREATING TEMP TABLE : temp_brk_glaggregated_d365
# ========================================

spark.sql(f"""CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_glaggregated_d365 (
		    DataSource STRING COLLATE UTF8_LCASE,
		    FiscalMonthDateKey INT,
		    PurchaseShipmentNumber STRING COLLATE UTF8_LCASE,
		    Cost DECIMAL(32,6),
		    SaleAmount DECIMAL(32,6)
		)""")



# ========================================
# INSERTING INTO TEMP TABLE(Store an aggregated version of the GL transactions at the fiscal month and shipment header level) : temp_brk_glstaging_d365
# ========================================

spark.sql(f"""INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_glaggregated_d365
    (DataSource, FiscalMonthDateKey, PurchaseShipmentNumber, Cost, SaleAmount)
    SELECT GL.DataSource, GL.FiscalMonthDateKey, GL.PurchaseShipmentNumber, 
           SUM(GL.Cost) AS Cost, SUM(GL.SaleAmount) AS SaleAmount
    FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 GL
    LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkvendtrans VT
        ON GL.VOUCHER = VT.Voucher COLLATE UTF8_LCASE 
        AND VT.IsDeleted = 0 
        AND VT.TRANSTYPE = 'VEND'
    WHERE GL.BusinessUnit <> '147000' AND VT.Voucher IS NULL
    GROUP BY GL.DataSource, GL.FiscalMonthDateKey, GL.PurchaseShipmentNumber""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# CREATING TEMP TABLE : temp_brk_ledgerjournal_d365
# ========================================


spark.sql(f"""CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
		    TransactionType STRING COLLATE UTF8_LCASE,
		    DataSource STRING COLLATE UTF8_LCASE,
		    FiscalMonth INT,
		    FiscalMonthDateKey INT,
		    TRANSDATE DATE,
		    ACCOUNTTYPE STRING COLLATE UTF8_LCASE,
		    ACCOUNTDISPLAYVALUE STRING COLLATE UTF8_LCASE,
		    OFFSETACCOUNTTYPE STRING COLLATE UTF8_LCASE, 
		    OFFSETACCOUNTDISPLAYVALUE STRING COLLATE UTF8_LCASE,
		    DESCRIPTION STRING COLLATE UTF8_LCASE,
		    TEXT STRING COLLATE UTF8_LCASE,
		    OFFSETTEXT STRING COLLATE UTF8_LCASE,
		    VOUCHER STRING COLLATE UTF8_LCASE,
		    VENDORNUMBER STRING COLLATE UTF8_LCASE,
		    CUSTOMERNUMBER STRING COLLATE UTF8_LCASE,
		    WEIGHT STRING COLLATE UTF8_LCASE,
		    CREDITAMOUNT DECIMAL(32,6),
		    DEBITAMOUNT DECIMAL(32,6),
		    GLAccountFirstDigit STRING COLLATE UTF8_LCASE,
		    GLAccount STRING COLLATE UTF8_LCASE,
		    BusinessUnit STRING COLLATE UTF8_LCASE,
			Office STRING COLLATE UTF8_LCASE,
		    SaleType STRING COLLATE UTF8_LCASE,
		    Cost DECIMAL(32,6),
		    SaleAmount DECIMAL(32,6),
			KeepRemove STRING COLLATE UTF8_LCASE,
			MANUALCUSTOMERNUMBER STRING COLLATE UTF8_LCASE )
		""")


# ========================================
# INSERTING INTO TEMP TABLE(Store a list of all the manually entered GL transactions) : temp_brk_ledgerjournal_d365
# ========================================


if MonthToLoad == 999999:
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
        TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
        OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
        VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT)
    SELECT 'Manual Journal Entry', TD.Fiscal_Month, CAST(GL.TRANSDATE AS DATE),
           GL.ACCOUNTTYPE, GL.ACCOUNTDISPLAYVALUE,
           GL.OFFSETACCOUNTTYPE, GL.OFFSETACCOUNTDISPLAYVALUE,
           GL.DESCRIPTION, GL.TEXT, GL.OFFSETTEXT,
           GL.VOUCHER, GL.VENDORNUMBER, GL.CUSTOMERNUMBER,
           GL.WEIGHT, GL.CREDITAMOUNT, GL.DEBITAMOUNT
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal GL
    JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
      ON CAST(GL.TRANSDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.COMPANY = 'BRK'
      AND TD.Fiscal_Month = {CurrentFiscalMonth}
    """)

    
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
        TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
        OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
        VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT)
    SELECT 'Offset Manual Journal Entry', TD.Fiscal_Month, CAST(GL.TRANSDATE AS DATE),
           GL.OFFSETACCOUNTTYPE, GL.OFFSETACCOUNTDISPLAYVALUE,
           GL.ACCOUNTTYPE, GL.ACCOUNTDISPLAYVALUE,
           GL.DESCRIPTION, GL.TEXT, GL.OFFSETTEXT,
           GL.VOUCHER, GL.VENDORNUMBER, GL.CUSTOMERNUMBER,
           GL.WEIGHT, GL.CREDITAMOUNT, GL.DEBITAMOUNT
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal GL
    JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
      ON CAST(GL.TRANSDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.COMPANY = 'BRK'
      AND GL.OFFSETACCOUNTDISPLAYVALUE <> ''
      AND TD.Fiscal_Month = {CurrentFiscalMonth}
    """)

elif MonthToLoad == 888888:
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
        TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
        OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
        VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT)
    SELECT 'Manual Journal Entry', TD.Fiscal_Month, CAST(GL.TRANSDATE AS DATE),
           GL.ACCOUNTTYPE, GL.ACCOUNTDISPLAYVALUE,
           GL.OFFSETACCOUNTTYPE, GL.OFFSETACCOUNTDISPLAYVALUE,
           GL.DESCRIPTION, GL.TEXT, GL.OFFSETTEXT,
           GL.VOUCHER, GL.VENDORNUMBER, GL.CUSTOMERNUMBER,
           GL.WEIGHT, GL.CREDITAMOUNT, GL.DEBITAMOUNT
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal GL
    JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
      ON CAST(GL.TRANSDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.COMPANY = 'BRK'
      AND TD.Fiscal_Month IN ({CurrentFiscalMonth}, {PriorFiscalMonth})
    """)

    
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
        TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
        OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
        VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT)
    SELECT 'Offset Manual Journal Entry', TD.Fiscal_Month, CAST(GL.TRANSDATE AS DATE),
           GL.OFFSETACCOUNTTYPE, GL.OFFSETACCOUNTDISPLAYVALUE,
           GL.ACCOUNTTYPE, GL.ACCOUNTDISPLAYVALUE,
           GL.DESCRIPTION, GL.TEXT, GL.OFFSETTEXT,
           GL.VOUCHER, GL.VENDORNUMBER, GL.CUSTOMERNUMBER,
           GL.WEIGHT, GL.CREDITAMOUNT, GL.DEBITAMOUNT
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal GL
    JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
      ON CAST(GL.TRANSDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.COMPANY = 'BRK'
      AND GL.OFFSETACCOUNTDISPLAYVALUE <> ''
      AND TD.Fiscal_Month IN ({CurrentFiscalMonth}, {PriorFiscalMonth})
    """)

elif MonthToLoad == 777777:
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
        TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
        OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
        VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT)
    SELECT 'Manual Journal Entry', TD.Fiscal_Month, CAST(GL.TRANSDATE AS DATE),
           GL.ACCOUNTTYPE, GL.ACCOUNTDISPLAYVALUE,
           GL.OFFSETACCOUNTTYPE, GL.OFFSETACCOUNTDISPLAYVALUE,
           GL.DESCRIPTION, GL.TEXT, GL.OFFSETTEXT,
           GL.VOUCHER, GL.VENDORNUMBER, GL.CUSTOMERNUMBER,
           GL.WEIGHT, GL.CREDITAMOUNT, GL.DEBITAMOUNT
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal GL
    JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
      ON CAST(GL.TRANSDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.COMPANY = 'BRK'
      AND TD.Fiscal_Month = {PriorFiscalMonth}
    """)

    
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
        TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
        OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
        VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT)
    SELECT 'Offset Manual Journal Entry', TD.Fiscal_Month, CAST(GL.TRANSDATE AS DATE),
           GL.OFFSETACCOUNTTYPE, GL.OFFSETACCOUNTDISPLAYVALUE,
           GL.ACCOUNTTYPE, GL.ACCOUNTDISPLAYVALUE,
           GL.DESCRIPTION, GL.TEXT, GL.OFFSETTEXT,
           GL.VOUCHER, GL.VENDORNUMBER, GL.CUSTOMERNUMBER,
           GL.WEIGHT, GL.CREDITAMOUNT, GL.DEBITAMOUNT
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal GL
    JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
      ON CAST(GL.TRANSDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.COMPANY = 'BRK'
      AND GL.OFFSETACCOUNTDISPLAYVALUE <> ''
      AND TD.Fiscal_Month = {PriorFiscalMonth}
    """)

elif MonthToLoad == 111111:
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
        TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
        OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
        VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT)
    SELECT 'Manual Journal Entry', TD.Fiscal_Month, CAST(GL.TRANSDATE AS DATE),
           GL.ACCOUNTTYPE, GL.ACCOUNTDISPLAYVALUE,
           GL.OFFSETACCOUNTTYPE, GL.OFFSETACCOUNTDISPLAYVALUE,
           GL.DESCRIPTION, GL.TEXT, GL.OFFSETTEXT,
           GL.VOUCHER, GL.VENDORNUMBER, GL.CUSTOMERNUMBER,
           GL.WEIGHT, GL.CREDITAMOUNT, GL.DEBITAMOUNT
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal GL
    JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
      ON CAST(GL.TRANSDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.COMPANY = 'BRK'
    """)

    
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
        TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
        OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
        VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT)
    SELECT 'Offset Manual Journal Entry', TD.Fiscal_Month, CAST(GL.TRANSDATE AS DATE),
           GL.OFFSETACCOUNTTYPE, GL.OFFSETACCOUNTDISPLAYVALUE,
           GL.ACCOUNTTYPE, GL.ACCOUNTDISPLAYVALUE,
           GL.DESCRIPTION, GL.TEXT, GL.OFFSETTEXT,
           GL.VOUCHER, GL.VENDORNUMBER, GL.CUSTOMERNUMBER,
           GL.WEIGHT, GL.CREDITAMOUNT, GL.DEBITAMOUNT
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal GL
    JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
      ON CAST(GL.TRANSDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.COMPANY = 'BRK'
      AND GL.OFFSETACCOUNTDISPLAYVALUE <> ''
    """)

else:
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
        TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
        OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
        VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT)
    SELECT 'Manual Journal Entry', TD.Fiscal_Month, CAST(GL.TRANSDATE AS DATE),
           GL.ACCOUNTTYPE, GL.ACCOUNTDISPLAYVALUE,
           GL.OFFSETACCOUNTTYPE, GL.OFFSETACCOUNTDISPLAYVALUE,
           GL.DESCRIPTION, GL.TEXT, GL.OFFSETTEXT,
           GL.VOUCHER, GL.VENDORNUMBER, GL.CUSTOMERNUMBER,
           GL.WEIGHT, GL.CREDITAMOUNT, GL.DEBITAMOUNT
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal GL
    JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
      ON CAST(GL.TRANSDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.COMPANY = 'BRK'
      AND TD.Fiscal_Month = {MonthToLoad}
    """)

   
    spark.sql(f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
        TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
        OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
        VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT)
    SELECT 'Offset Manual Journal Entry', TD.Fiscal_Month, CAST(GL.TRANSDATE AS DATE),
           GL.OFFSETACCOUNTTYPE, GL.OFFSETACCOUNTDISPLAYVALUE,
           GL.ACCOUNTTYPE, GL.ACCOUNTDISPLAYVALUE,
           GL.DESCRIPTION, GL.TEXT, GL.OFFSETTEXT,
           GL.VOUCHER, GL.VENDORNUMBER, GL.CUSTOMERNUMBER,
           GL.WEIGHT, GL.CREDITAMOUNT, GL.DEBITAMOUNT
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal GL
    JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
      ON CAST(GL.TRANSDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    WHERE GL.IsDeleted = 0
      AND GL.COMPANY = 'BRK'
      AND GL.OFFSETACCOUNTDISPLAYVALUE <> ''
      AND TD.Fiscal_Month = {MonthToLoad}
    """)

In [0]:
# ========================================
# CREATING TEMP VIEW : Reversals
# ========================================

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW Reversals AS
SELECT DISTINCT
    a.VOUCHER AS Voucher,
    b.TraceNum AS TraceNum,
    d.Fiscal_Month AS FiscalMonth,
    d.Date_Day
FROM {bronze_catalog}.{SourceSchemaName}.{SourceTableName} a
INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brktransactionreversaltrans b
    ON a.GENERALJOURNALACCOUNTENTRYRECID = b.RefRecId
    AND a.LEDGERNAME = b.DataAreaId
    AND b.IsDeleted = 0
INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date d
    ON CAST(a.ACCOUNTINGDATE AS DATE) = d.Date_Day
LEFT JOIN {enriched_catalog}.{temp_schema}.temp_brk_mgbushnell_d365 tmp
    ON a.GENERALJOURNALACCOUNTENTRYRECID = tmp.GENERALJOURNALACCOUNTENTRYRECID
WHERE a.IsDeleted = 0
    AND a.LEDGERNAME = 'BRK'
    AND COALESCE(b.Reversed, '') = 'Yes'
    AND COALESCE(b.TraceNum, '') <> ''
    AND tmp.GENERALJOURNALACCOUNTENTRYRECID IS NULL
""")



# ========================================
# CREATING AND POPULATING TEMP TABLE: temp_brk_genkeep_d365
# ========================================

spark.sql(f"""
CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_genkeep_d365 AS
SELECT DISTINCT
    LJ.VOUCHER AS InitialVoucher,
    RV.Date_Day AS InitialDate,
    RVf.Date_Day AS ReverseDate,
    RVf.FiscalMonth,
    LJ.TransactionType
FROM {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 LJ
INNER JOIN Reversals RV
    ON LJ.VOUCHER = RV.Voucher COLLATE UTF8_LCASE
INNER JOIN Reversals RVf
    ON RV.TraceNum = RVf.TraceNum COLLATE UTF8_LCASE
    AND RV.Voucher <> RVf.Voucher COLLATE UTF8_LCASE
    AND RV.FiscalMonth <> RVf.FiscalMonth
""")



# ========================================
# Insert reversal records with swapped credit/debit and KeepRemove = 'Keep'
# ========================================

spark.sql(f"""
INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
    TransactionType, FiscalMonth, TRANSDATE, ACCOUNTTYPE, ACCOUNTDISPLAYVALUE,
    OFFSETACCOUNTTYPE, OFFSETACCOUNTDISPLAYVALUE, DESCRIPTION, TEXT, OFFSETTEXT,
    VOUCHER, VENDORNUMBER, CUSTOMERNUMBER, WEIGHT, CREDITAMOUNT, DEBITAMOUNT, KeepRemove)
SELECT DISTINCT
    a.TransactionType,
    b.FiscalMonth,
    b.ReverseDate,
    a.ACCOUNTTYPE,
    a.ACCOUNTDISPLAYVALUE,
    a.OFFSETACCOUNTTYPE,
    a.OFFSETACCOUNTDISPLAYVALUE,
    a.DESCRIPTION,
    a.TEXT,
    a.OFFSETTEXT,
    a.VOUCHER,
    a.VENDORNUMBER,
    a.CUSTOMERNUMBER,
    a.WEIGHT,
    a.DEBITAMOUNT AS CreditReversed,
    a.CREDITAMOUNT AS DebitReversed,
    'Keep' AS KeepRemove
FROM {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 a
INNER JOIN {enriched_catalog}.{temp_schema}.temp_brk_genkeep_d365 b
    ON a.VOUCHER = b.InitialVoucher
    AND a.TransactionType = b.TransactionType
    AND CAST(a.TRANSDATE AS DATE) = b.InitialDate
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# CREATING TEMP TABLE: temp_brk_genremove_d365
# ========================================

spark.sql(f"""
CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_genremove_d365 (
    InitialVoucher STRING COLLATE UTF8_LCASE,
    InitialDate DATE,
    ReverseDate DATE
)
""")



# ========================================
# INSERTING INTO TEMP TABLE: temp_brk_genremove_d365
# ========================================

spark.sql(f"""
INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_genremove_d365
SELECT DISTINCT
    LJ.VOUCHER AS InitialVoucher,
    RV.Date_Day AS InitialDate,
    RVf.Date_Day AS ReverseDate
FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkledgerjournal LJ
INNER JOIN Reversals RV
    ON LJ.VOUCHER = RV.Voucher COLLATE UTF8_LCASE
INNER JOIN Reversals RVf
    ON RV.TraceNum = RVf.TraceNum
   AND RV.Voucher <> RVf.Voucher
   AND RV.FiscalMonth = RVf.FiscalMonth
""")



# ========================================
# Insert reversed copies flagged for removal
# ========================================

spark.sql(f"""INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 (
				TransactionType,
				FiscalMonth,
				TRANSDATE,
				ACCOUNTTYPE,
				ACCOUNTDISPLAYVALUE,
				OFFSETACCOUNTTYPE,
				OFFSETACCOUNTDISPLAYVALUE,
				DESCRIPTION,
				TEXT,
				OFFSETTEXT,
				VOUCHER,
				VENDORNUMBER,
				CUSTOMERNUMBER,
				WEIGHT,
				CREDITAMOUNT,
				DEBITAMOUNT,
				KeepRemove
			)
    SELECT DISTINCT a.TransactionType, a.FiscalMonth, a.TRANSDATE,
        a.ACCOUNTTYPE, a.ACCOUNTDISPLAYVALUE, a.OFFSETACCOUNTTYPE, a.OFFSETACCOUNTDISPLAYVALUE,
        a.DESCRIPTION, a.TEXT, a.OFFSETTEXT, a.VOUCHER, a.VENDORNUMBER, a.CUSTOMERNUMBER, a.WEIGHT,
        a.CREDITAMOUNT, a.DEBITAMOUNT, 'Remove'
    FROM {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 a
    INNER JOIN {enriched_catalog}.{temp_schema}.temp_brk_genremove_d365 b ON a.VOUCHER = b.InitialVoucher AND CAST(a.TRANSDATE AS DATE) = b.ReverseDate""")



# ========================================
# Delete flagged records
# ========================================

spark.sql(f"DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 WHERE KeepRemove = 'Remove'")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# Set records to keep in the ledger table
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365
SET KeepRemove = 'Keep'
""")



# ========================================
# Remove manual journal entries tied to shipments
# ========================================

spark.sql(f"""
DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365
WHERE get(split(cast(ACCOUNTDISPLAYVALUE as string), '-'),6) <> ''
""")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 SET GLAccountFirstDigit = LEFT(split(cast(ACCOUNTDISPLAYVALUE as string), '-')[0],1)""")
spark.sql(f"DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 WHERE GLAccountFirstDigit NOT IN ('4','5')")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 SET BusinessUnit = split(cast(ACCOUNTDISPLAYVALUE as string), '-')[2]""")
spark.sql(f"DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 WHERE BusinessUnit = '147000'")



spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 SET GLAccount = split(cast(ACCOUNTDISPLAYVALUE as string), '-')[0]""")
spark.sql(f"DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 WHERE GLAccount IN ('51120','51125','51130')")



# ========================================
# if the transaction occurred in the current fiscal month, then it is considered "Summary"; if it occurred in a prior fiscal month, then it is considered "History"
# ========================================

spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 SET DataSource = CASE WHEN FiscalMonth = {CurrentFiscalMonth} THEN 'Summary' ELSE 'History' END""")
spark.sql(f"""MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 GL USING {enriched_catalog}.{temp_schema}.temp_brk_fiscalmonthdates_d365 TD2
    ON GL.FiscalMonth = TD2.Fiscal_Month WHEN MATCHED THEN UPDATE SET GL.FiscalMonthDateKey = TD2.DateKey""")



spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 SET Office = split(cast(ACCOUNTDISPLAYVALUE as string), '-')[3]""")


spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 SET SaleType = split(cast(ACCOUNTDISPLAYVALUE as string), '-')[5]""")




# ========================================
# if the user entered the customer number in the financial dimension rather than the customer number field, then pull it from there
# ========================================

spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365
    SET CUSTOMERNUMBER = OFFSETACCOUNTDISPLAYVALUE
    WHERE COALESCE(CUSTOMERNUMBER,'') = '' AND OFFSETACCOUNTTYPE = 'Cust'
    AND TransactionType = 'Offset Manual Journal Entry'""")



spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365
    SET Cost = COALESCE(DEBITAMOUNT,0) + (COALESCE(CREDITAMOUNT,0)*-1)
    WHERE GLAccountFirstDigit = '5' AND (DEBITAMOUNT IS NOT NULL OR CREDITAMOUNT IS NOT NULL) AND TransactionType = 'Manual Journal Entry' """)



spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365
    SET Cost = (COALESCE(DEBITAMOUNT,0)*-1) + COALESCE(CREDITAMOUNT,0)
    WHERE GLAccountFirstDigit = '5' AND (DEBITAMOUNT IS NOT NULL OR CREDITAMOUNT IS NOT NULL) AND TransactionType = 'Offset Manual Journal Entry' """)



spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365
    SET SaleAmount = COALESCE(CREDITAMOUNT,0) + (COALESCE(DEBITAMOUNT,0)*-1)
    WHERE GLAccountFirstDigit = '4' AND (DEBITAMOUNT IS NOT NULL OR CREDITAMOUNT IS NOT NULL) AND TransactionType = 'Manual Journal Entry' """)




spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365
    SET SaleAmount = (COALESCE(CREDITAMOUNT,0)*-1) + COALESCE(DEBITAMOUNT,0)
    WHERE GLAccountFirstDigit = '4' AND (DEBITAMOUNT IS NOT NULL OR CREDITAMOUNT IS NOT NULL) AND TransactionType = 'Offset Manual Journal Entry' """)



# ========================================
# SET THE MANUAL CUSTOMER NUMBER
# ========================================

spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365
    SET MANUALCUSTOMERNUMBER = CASE
        WHEN ACCOUNTTYPE = 'Cust' AND INSTR(CAST(ACCOUNTDISPLAYVALUE AS STRING), '-') = 0 THEN CAST(ACCOUNTDISPLAYVALUE AS STRING)
        WHEN OFFSETACCOUNTTYPE = 'Cust' AND INSTR(CAST(OFFSETACCOUNTDISPLAYVALUE AS STRING), '-') = 0 THEN CAST(OFFSETACCOUNTDISPLAYVALUE AS STRING)
        WHEN COALESCE(CUSTOMERNUMBER, '0') <> '0' THEN CUSTOMERNUMBER
        ELSE '' END""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# CREATING TEMP TABLE: temp_brk_purchinvline_d365
# ========================================

spark.sql(f"""CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_purchinvline_d365(
           FiscalMonth INT,
		   FiscalMonthDateKey INT,
		   SHIPMENTID INT,
		   InvoicedQuantity DECIMAL(32,6) )
""")



# ========================================
# INSERTING INTO TEMP TABLE: temp_brk_purchinvline_d365
# ========================================

# Columns to insert
columns = "FiscalMonth, SHIPMENTID, InvoicedQuantity"


if MonthToLoad == 999999:  
    spark.sql(f"""
        INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_purchinvline_d365 ({columns})
        SELECT 
            TD.Fiscal_Month AS FiscalMonth,
            PK.SHIPMENTID,
            SUM(PInvL.InvoicedQuantity) AS InvoicedQuantity
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkpurchaseinvoiceline PInvL
        INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
            ON CAST(PInvL.INVOICEDATE AS DATE) = CAST(TD.Date_Day AS DATE)
        INNER JOIN (
            SELECT DISTINCT PURCHID, SHIPMENTID
            FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinepurchasekeys
            WHERE DATAAREAID = 'BRK'
              AND IsDeleted = 0
        ) PK
            ON PInvL.PurchId = PK.PURCHID
        WHERE PInvL.DATAAREAID = 'BRK'
          AND PInvL.IsDeleted = 0
          AND TD.Fiscal_Month = {CurrentFiscalMonth}
        GROUP BY TD.Fiscal_Month, PK.SHIPMENTID
    """)

elif MonthToLoad == 888888:  
    spark.sql(f"""
        INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_purchinvline_d365 ({columns})
        SELECT 
            TD.Fiscal_Month,
            PK.SHIPMENTID,
            SUM(PInvL.InvoicedQuantity)
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkpurchaseinvoiceline PInvL
        INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
            ON CAST(PInvL.INVOICEDATE AS DATE) = CAST(TD.Date_Day AS DATE)
        INNER JOIN (
            SELECT DISTINCT PURCHID, SHIPMENTID
            FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinePurchaseKeys
            WHERE DATAAREAID = 'BRK'
              AND IsDeleted = 0
        ) PK
            ON PInvL.PurchId = PK.PURCHID
        WHERE PInvL.DATAAREAID = 'BRK'
          AND PInvL.IsDeleted = 0
          AND TD.Fiscal_Month IN ({CurrentFiscalMonth},{PriorFiscalMonth})
        GROUP BY TD.Fiscal_Month, PK.SHIPMENTID
    """)

elif MonthToLoad == 777777:  
    spark.sql(f"""
        INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_purchinvline_d365 ({columns})
        SELECT 
            TD.Fiscal_Month,
            PK.SHIPMENTID,
            SUM(PInvL.InvoicedQuantity)
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkpurchaseinvoiceline PInvL
        INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
            ON CAST(PInvL.INVOICEDATE AS DATE) = CAST(TD.Date_Day AS DATE)
        INNER JOIN (
            SELECT DISTINCT PURCHID, SHIPMENTID
            FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinePurchaseKeys
            WHERE DATAAREAID = 'BRK'
              AND IsDeleted = 0
        ) PK
            ON PInvL.PurchId = PK.PURCHID
        WHERE PInvL.DATAAREAID = 'BRK'
          AND PInvL.IsDeleted = 0
          AND TD.Fiscal_Month = {PriorFiscalMonth}
        GROUP BY TD.Fiscal_Month, PK.SHIPMENTID
    """)

elif MonthToLoad == 111111:  
    spark.sql(f"""
        INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_purchinvline_d365 ({columns})
        SELECT 
            TD.Fiscal_Month,
            PK.SHIPMENTID,
            SUM(PInvL.InvoicedQuantity)
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkpurchaseinvoiceline PInvL
        INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
            ON CAST(PInvL.INVOICEDATE AS DATE) = CAST(TD.Date_Day AS DATE)
        INNER JOIN (
            SELECT DISTINCT PURCHID, SHIPMENTID
            FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinePurchaseKeys
            WHERE DATAAREAID = 'BRK'
              AND IsDeleted = 0
        ) PK
            ON PInvL.PurchId = PK.PURCHID
        WHERE PInvL.DATAAREAID = 'BRK'
          AND PInvL.IsDeleted = 0
        GROUP BY TD.Fiscal_Month, PK.SHIPMENTID
    """)

else:  
    spark.sql(f"""
        INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_purchinvline_d365 ({columns})
        SELECT 
            TD.Fiscal_Month,
            PK.SHIPMENTID,
            SUM(PInvL.InvoicedQuantity)
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkpurchaseinvoiceline PInvL
        INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
            ON CAST(PInvL.INVOICEDATE AS DATE) = CAST(TD.Date_Day AS DATE)
        INNER JOIN (
            SELECT DISTINCT PURCHID, SHIPMENTID
            FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinePurchaseKeys
            WHERE DATAAREAID = 'BRK'
              AND IsDeleted = 0
        ) PK
            ON PInvL.PurchId = PK.PURCHID
        WHERE PInvL.DATAAREAID = 'BRK'
          AND PInvL.IsDeleted = 0
          AND TD.Fiscal_Month = {MonthToLoad}
        GROUP BY TD.Fiscal_Month, PK.SHIPMENTID
    """)

In [0]:
# ========================================
# the last day of the fiscal month that the transaction occurred in
# ========================================

spark.sql(f"""
          MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_purchinvline_d365 AS PInvL
USING {enriched_catalog}.{temp_schema}.temp_brk_fiscalmonthdates_d365 AS TD2
ON PInvL.FiscalMonth = TD2.Fiscal_Month
WHEN MATCHED THEN
  UPDATE SET PInvL.FiscalMonthDateKey = TD2.DateKey
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# CREATING TEMP TABLE: temp_brk_salesinvline_d365
# ========================================

spark.sql(f"""CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365 (FiscalMonth INT,
		    FiscalMonthDateKey INT,
		    SHIPMENTID INT,
		    INVOICEDQUANTITY DECIMAL(32,6)
		)
""")



# ========================================
# INSERTING INTO TEMP TABLE: temp_brk_salesinvline_d365
# ========================================

if MonthToLoad == 999999:
    spark.sql( f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365 (FiscalMonth, SHIPMENTID, INVOICEDQUANTITY)
    SELECT TD.Fiscal_Month AS FiscalMonth,
           SK.SHIPMENTID,
           SUM(SInvL.INVOICEDQUANTITY) AS INVOICEDQUANTITY
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoicelinev2 SInvL
    INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
        ON CAST(SInvL.INVOICEDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoiceheaderv2 SInvH
        ON SInvL.INVOICENUMBER = SInvH.INVOICENUMBER
       AND SInvL.LEDGERVOUCHER = SInvH.LEDGERVOUCHER
       AND SInvL.INVOICEDATE = SInvH.INVOICEDATE
       AND SInvH.DATAAREAID = 'BRK'
       AND SInvH.IsDeleted = 0
    INNER JOIN (
        SELECT DISTINCT SALESID, SHIPMENTID
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinesaleskeys
        WHERE DATAAREAID = 'BRK'
          AND IsDeleted = 0
    ) SK ON SInvH.SALESORDERNUMBER = SK.SALESID
    WHERE SInvL.DATAAREAID = 'BRK'
      AND SInvL.IsDeleted = 0
      AND TD.Fiscal_Month = {CurrentFiscalMonth}
    GROUP BY TD.Fiscal_Month, SK.SHIPMENTID
    """)


elif MonthToLoad == 888888:
    spark.sql( f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365 (FiscalMonth, SHIPMENTID, INVOICEDQUANTITY)
    SELECT TD.Fiscal_Month AS FiscalMonth,
           SK.SHIPMENTID,
           SUM(SInvL.INVOICEDQUANTITY) AS INVOICEDQUANTITY
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoicelinev2 SInvL
    INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
        ON CAST(SInvL.INVOICEDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoiceheaderv2 SInvH
        ON SInvL.INVOICENUMBER = SInvH.INVOICENUMBER
       AND SInvL.LEDGERVOUCHER = SInvH.LEDGERVOUCHER
       AND SInvL.INVOICEDATE = SInvH.INVOICEDATE
       AND SInvH.DATAAREAID = 'BRK'
       AND SInvH.IsDeleted = 0
    INNER JOIN (
        SELECT DISTINCT SALESID, SHIPMENTID
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinesaleskeys
        WHERE DATAAREAID = 'BRK'
          AND IsDeleted = 0
    ) SK ON SInvH.SALESORDERNUMBER = SK.SALESID
    WHERE SInvL.DATAAREAID = 'BRK'
      AND SInvL.IsDeleted = 0
      AND TD.Fiscal_Month IN ({CurrentFiscalMonth},{PriorFiscalMonth})
    GROUP BY TD.Fiscal_Month, SK.SHIPMENTID
    """)


elif MonthToLoad == 777777:
    spark.sql( f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365 (FiscalMonth, SHIPMENTID, INVOICEDQUANTITY)
    SELECT TD.Fiscal_Month AS FiscalMonth,
           SK.SHIPMENTID,
           SUM(SInvL.INVOICEDQUANTITY) AS INVOICEDQUANTITY
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoicelinev2 SInvL
    INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
        ON CAST(SInvL.INVOICEDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoiceheaderv2 SInvH
        ON SInvL.INVOICENUMBER = SInvH.INVOICENUMBER
       AND SInvL.LEDGERVOUCHER = SInvH.LEDGERVOUCHER
       AND SInvL.INVOICEDATE = SInvH.INVOICEDATE
       AND SInvH.DATAAREAID = 'BRK'
       AND SInvH.IsDeleted = 0
    INNER JOIN (
        SELECT DISTINCT SALESID, SHIPMENTID
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinesaleskeys
        WHERE DATAAREAID = 'BRK'
          AND IsDeleted = 0
    ) SK ON SInvH.SALESORDERNUMBER = SK.SALESID
    WHERE SInvL.DATAAREAID = 'BRK'
      AND SInvL.IsDeleted = 0
      AND TD.Fiscal_Month = {PriorFiscalMonth}
    GROUP BY TD.Fiscal_Month, SK.SHIPMENTID
    """)


elif MonthToLoad == 111111:
    spark.sql( f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365 (FiscalMonth, SHIPMENTID, INVOICEDQUANTITY)
    SELECT TD.Fiscal_Month AS FiscalMonth,
           SK.SHIPMENTID,
           SUM(SInvL.INVOICEDQUANTITY) AS INVOICEDQUANTITY
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoicelinev2 SInvL
    INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
        ON CAST(SInvL.INVOICEDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoiceheaderv2 SInvH
        ON SInvL.INVOICENUMBER = SInvH.INVOICENUMBER
       AND SInvL.LEDGERVOUCHER = SInvH.LEDGERVOUCHER
       AND SInvL.INVOICEDATE = SInvH.INVOICEDATE
       AND SInvH.DATAAREAID = 'BRK'
       AND SInvH.IsDeleted = 0
    INNER JOIN (
        SELECT DISTINCT SALESID, SHIPMENTID
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinesaleskeys
        WHERE DATAAREAID = 'BRK'
          AND IsDeleted = 0
    ) SK ON SInvH.SALESORDERNUMBER = SK.SALESID
    WHERE SInvL.DATAAREAID = 'BRK'
      AND SInvL.IsDeleted = 0
    GROUP BY TD.Fiscal_Month, SK.SHIPMENTID
    """)


else:
    spark.sql( f"""
    INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365 (FiscalMonth, SHIPMENTID, INVOICEDQUANTITY)
    SELECT TD.Fiscal_Month AS FiscalMonth,
           SK.SHIPMENTID,
           SUM(SInvL.INVOICEDQUANTITY) AS INVOICEDQUANTITY
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoicelinev2 SInvL
    INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date TD
        ON CAST(SInvL.INVOICEDATE AS DATE) = CAST(TD.Date_Day AS DATE)
    INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoiceheaderv2 SInvH
        ON SInvL.INVOICENUMBER = SInvH.INVOICENUMBER
       AND SInvL.LEDGERVOUCHER = SInvH.LEDGERVOUCHER
       AND SInvL.INVOICEDATE = SInvH.INVOICEDATE
       AND SInvH.DATAAREAID = 'BRK'
       AND SInvH.IsDeleted = 0
    INNER JOIN (
        SELECT DISTINCT SALESID, SHIPMENTID
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinesaleskeys
        WHERE DATAAREAID = 'BRK'
          AND IsDeleted = 0
    ) SK ON SInvH.SALESORDERNUMBER = SK.SALESID
    WHERE SInvL.DATAAREAID = 'BRK'
      AND SInvL.IsDeleted = 0
      AND TD.Fiscal_Month = {MonthToLoad}
    GROUP BY TD.Fiscal_Month, SK.SHIPMENTID
    """)

In [0]:
# ========================================
# the last day of the fiscal month that the transaction occurred in
# ========================================

spark.sql( f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365 SInvL
USING {enriched_catalog}.{temp_schema}.temp_brk_fiscalmonthdates_d365 TD2 ON SInvL.FiscalMonth = TD2.Fiscal_Month WHEN MATCHED THEN UPDATE SET SInvL.FiscalMonthDateKey = TD2.DateKey
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# CREATING TEMP TABLE: temp_brk_factprofit_d365
# ========================================

spark.sql(f"""CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 (
    TransactionType STRING COLLATE UTF8_LCASE,
    f_OriginalID INT,
    f_DataSource STRING COLLATE UTF8_LCASE,
    f_SourceSystemKey BIGINT,
    f_FiscalMonthDateKey INT,
    f_PurchaseShipmentNumber DOUBLE,
    f_PurchaseShipmentLine INT,
    f_PurchaseReference STRING COLLATE UTF8_LCASE,
    f_SaleReference STRING COLLATE UTF8_LCASE,
    f_ServiceItem STRING COLLATE UTF8_LCASE,
    f_BuyPlanMonthKey INT,
    f_BrokerageGroupKey BIGINT,
    f_SaleTypeKey BIGINT,
    f_PurchaseGradeKey BIGINT,
    f_SaleGradeKey BIGINT,
    f_ServiceItemKey BIGINT,
    f_VehicleTypeKey BIGINT,
    f_SupplierAddressBookKey BIGINT,
    f_OriginLocationKey BIGINT,
    f_ConsumerKey BIGINT,
    f_DestinationLocationKey BIGINT,
    f_PurchaseOfficeKey BIGINT,
    f_SaleOfficeKey BIGINT,
    f_ProfitPropertiesKey BIGINT,
    f_ShipmentPropertiesKey BIGINT,
    f_OriginTicketNumber STRING COLLATE UTF8_LCASE,
    f_MillTicketDate TIMESTAMP,
    f_MillTicketNumber STRING COLLATE UTF8_LCASE,
    f_PurchaseContractDate TIMESTAMP,
    f_SaleEffectiveDate TIMESTAMP,
    f_SaleShipmentNumber STRING COLLATE UTF8_LCASE,
    f_PurchaseNumber STRING COLLATE UTF8_LCASE,
    f_PurchaseLine DOUBLE,
    f_SaleNumber DOUBLE,
    f_BOLNumber STRING COLLATE UTF8_LCASE,
    f_MillNetWgt INT,
    f_PurchasedWgt DOUBLE,
    f_SaleWgt DOUBLE,
    f_SaleAmount DOUBLE,
    f_Margin DOUBLE,
    f_Cost DOUBLE,
    f_DJJGeneratedFlag INT,
    f_EnrichedTimestampUTC TIMESTAMP,
    lkp_PURCHASECONTRACTID INT,
    lkp_PURCHASELINE DECIMAL(32,6),
    lkp_SALESCONTRACTID INT,
    lkp_SALESLINE DECIMAL(32,6),
    lkp_ITEMCODE STRING COLLATE UTF8_LCASE,
    lkp_MODE STRING COLLATE UTF8_LCASE,
    lkp_ORIGIN BIGINT,
    lkp_DESTINATION BIGINT,
    lkp_SALESTYPE STRING COLLATE UTF8_LCASE,
    lkp_VENDORNUMBER STRING COLLATE UTF8_LCASE,
    lkp_MANUALCUSTOMERNUMBER STRING COLLATE UTF8_LCASE,
    lkp_LEDGERVOUCHER STRING COLLATE UTF8_LCASE,
    lkp_INVOICENUMBER STRING COLLATE UTF8_LCASE,
    lkp_OFFICE STRING COLLATE UTF8_LCASE,
    lkp_BUSINESSUNIT STRING COLLATE UTF8_LCASE
)
""")

DataFrame[]

In [0]:
# ========================================
# first section grabs shipments that have cost or sale journal entries against them and summarizes them by the primary keys
# ========================================

spark.sql(f"""INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 (
            TransactionType, 
            f_DataSource, 
            f_FiscalMonthDateKey, 
            f_PurchaseShipmentNumber, 
            f_PurchaseShipmentLine, 
            f_PurchaseReference, 
            f_SaleReference, 
            f_OriginTicketNumber,
            f_MillTicketDate,
            f_MillTicketNumber,
            f_SaleShipmentNumber,
            f_PurchaseNumber,
            f_PurchaseLine,
            f_SaleNumber,
            f_BOLNumber,
            f_SaleAmount,
            f_Cost,
            lkp_PURCHASECONTRACTID, 
            lkp_PURCHASELINE,
            lkp_SALESCONTRACTID,
            lkp_SALESLINE,
            lkp_ITEMCODE,
            lkp_MODE,
            lkp_ORIGIN,
            lkp_DESTINATION,
            lkp_SALESTYPE,
            lkp_VENDORNUMBER,
            lkp_MANUALCUSTOMERNUMBER
        )
    SELECT 'Shipment' AS TransactionType, GL.DataSource AS f_DataSource, GL.FiscalMonthDateKey AS f_FiscalMonthDateKey,
        SH.SHIPMENTID AS f_PurchaseShipmentNumber, SL.BRKSHIPMENTLINE AS f_PurchaseShipmentLine,
        CAST(CAST(date_format(SH.SHIPPEDDATE, 'yyyyMMdd') AS INT) AS STRING) || ' ' || SH.CARNUMBER AS f_PurchaseReference,
        CAST(CAST(date_format(SH.SHIPPEDDATE, 'yyyyMMdd') AS INT) AS STRING) || ' ' || SH.CARNUMBER AS f_SaleReference,
        SL.ORIGINTICKETNUMBER AS f_OriginTicketNumber,
        CAST(SL.DESTINATIONTICKETDATE AS DATE) AS f_MillTicketDate, SL.DESTINATIONTICKETNUMBER AS f_MillTicketNumber,
        CAST(SH.SHIPMENTID AS STRING) AS f_SaleShipmentNumber,
        CAST(SL.PURCHASECONTRACTID AS STRING) AS f_PurchaseNumber, SL.PURCHASELINE AS f_PurchaseLine,
        SL.SALESCONTRACTID AS f_SaleNumber, COALESCE(SH.BILLOFLADINGNUMBER,'') AS f_BOLNumber,
        GL.SaleAmount AS f_SaleAmount, GL.Cost AS f_Cost,
        SL.PURCHASECONTRACTID AS lkp_PURCHASECONTRACTID, SL.PURCHASELINE AS lkp_PURCHASELINE,
        SL.SALESCONTRACTID AS lkp_SALESCONTRACTID, SL.SALESLINE AS lkp_SALESLINE,
        SL.ITEMCODE AS lkp_ITEMCODE, SL.MODE AS lkp_MODE, 
        SL.ORIGIN AS lkp_ORIGIN, 
        SL.DESTINATION AS lkp_DESTINATION,
        SL.SALESTYPE AS lkp_SALESTYPE, SL.VENDORNUMBER AS lkp_VENDORNUMBER, SL.CUSTOMERNUMBER AS lkp_MANUALCUSTOMERNUMBER
    FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentheader SH
    INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentline SL
        ON SH.SHIPMENTID = SL.SHIPMENTID AND SL.BRKSHIPMENTLINE = 1
        AND SL.DATAAREAID = 'BRK' AND SL.IsDeleted = 0
    INNER JOIN {enriched_catalog}.{temp_schema}.temp_brk_glaggregated_d365 GL ON SH.SHIPMENTID = TRY_CAST(GL.PurchaseShipmentNumber AS INT)
    WHERE SH.IsDeleted = 0 AND SH.DATAAREAID = 'BRK'
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# second section grabs manual sale orders (i.e. those that were not generated by a shipment) and summarizes them by the primary keys
# ========================================

spark.sql(f"""INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
(TransactionType, f_DataSource, f_FiscalMonthDateKey, f_PurchaseShipmentNumber, f_PurchaseShipmentLine,
 f_PurchaseReference, f_SaleReference, f_MillTicketDate, f_MillTicketNumber, f_SaleShipmentNumber,
 f_PurchaseNumber, f_PurchaseLine, f_SaleNumber, f_SaleAmount, f_Cost, lkp_SALESTYPE, lkp_MANUALCUSTOMERNUMBER, lkp_OFFICE)
SELECT 'Manual Sale Order', CASE WHEN SD.Fiscal_Month = {CurrentFiscalMonth} THEN 'Summary' ELSE 'History' END,
       SD2.DateKey, 0, 0, '',
       CASE WHEN COALESCE(SO.CUSTOMERSORDERREFERENCE,'') = '' THEN COALESCE(SO.SALESORDERNUMBER,'') ELSE COALESCE(SO.CUSTOMERSORDERREFERENCE,'') END,
       CAST('1900-01-01' AS DATE), '', '0', '0', 0, 0,
       SUM(Sale.SaleAmt), SUM(Cost.Cost), MAX(GL.SaleType), SO.ORDERINGCUSTOMERACCOUNTNUMBER,
       SPLIT(CAST(SO.DEFAULTLEDGERDIMENSIONDISPLAYVALUE AS STRING), '-')[2]
FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesorderheaderv2 SO
LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinesaleskeys SKeys 
       ON SO.SALESORDERNUMBER = SKeys.SALESID AND SKeys.DATAAREAID = 'BRK' AND SKeys.IsDeleted = 0
INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoiceheaderv2 SInvH 
       ON SO.SALESORDERNUMBER = SInvH.SALESORDERNUMBER AND SInvH.DATAAREAID = 'BRK' AND SInvH.IsDeleted = 0
INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoicelinev2 SInvL 
       ON SInvL.INVOICENUMBER = SInvH.INVOICENUMBER 
       AND SInvL.LEDGERVOUCHER = SInvH.LEDGERVOUCHER 
       AND SInvL.INVOICEDATE = SInvH.INVOICEDATE 
       AND SInvL.DATAAREAID = 'BRK' AND SInvL.IsDeleted = 0
INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date SD 
       ON CAST(SInvL.INVOICEDATE AS DATE) = CAST(SD.Date_Day AS DATE)
INNER JOIN {enriched_catalog}.{temp_schema}.temp_brk_fiscalmonthdates_d365 SD2 
       ON SD.Fiscal_Month = SD2.Fiscal_Month
LEFT OUTER JOIN (SELECT VOUCHER, DOCUMENTNUMBER, MAX(SaleType) AS SaleType FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 GROUP BY VOUCHER, DOCUMENTNUMBER) GL 
       ON SInvH.LEDGERVOUCHER COLLATE UTF8_LCASE = GL.VOUCHER AND SInvH.INVOICENUMBER COLLATE UTF8_LCASE = GL.DOCUMENTNUMBER
LEFT OUTER JOIN (SELECT VOUCHER, DOCUMENTNUMBER, SUM(TRANSACTIONCURRENCYAMOUNT)*-1 AS SaleAmt FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 WHERE GLAccountFirstDigit = '4' GROUP BY VOUCHER, DOCUMENTNUMBER) Sale 
       ON SInvH.LEDGERVOUCHER COLLATE UTF8_LCASE = Sale.VOUCHER AND SInvH.INVOICENUMBER COLLATE UTF8_LCASE = Sale.DOCUMENTNUMBER
LEFT OUTER JOIN (SELECT VOUCHER, DOCUMENTNUMBER, SUM(TRANSACTIONCURRENCYAMOUNT)*-1 AS Cost FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 WHERE GLAccountFirstDigit = '5' GROUP BY VOUCHER, DOCUMENTNUMBER) Cost 
       ON SInvH.LEDGERVOUCHER COLLATE UTF8_LCASE = Cost.VOUCHER AND SInvH.INVOICENUMBER COLLATE UTF8_LCASE = Cost.DOCUMENTNUMBER
WHERE SO.IsDeleted = 0 AND SO.DATAAREAID = 'BRK' AND SKeys.SALESID IS NULL
      AND (({MonthToLoad} = 999999 AND SD.Fiscal_Month = {CurrentFiscalMonth}) 
           OR ({MonthToLoad} = 888888 AND SD.Fiscal_Month IN ({CurrentFiscalMonth},{PriorFiscalMonth})) 
           OR ({MonthToLoad} = 777777 AND SD.Fiscal_Month = {PriorFiscalMonth}) 
           OR {MonthToLoad} = 111111 
           OR {MonthToLoad} = SD.Fiscal_Month)
GROUP BY SD2.DateKey, CASE WHEN SD.Fiscal_Month = {CurrentFiscalMonth} THEN 'Summary' ELSE 'History' END,
         CASE WHEN COALESCE(SO.CUSTOMERSORDERREFERENCE,'') = '' THEN COALESCE(SO.SALESORDERNUMBER,'') ELSE COALESCE(SO.CUSTOMERSORDERREFERENCE,'') END,
         GL.SaleType, SO.ORDERINGCUSTOMERACCOUNTNUMBER, SPLIT(CAST(SO.DEFAULTLEDGERDIMENSIONDISPLAYVALUE AS STRING), '-')[2]
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# third section grabs manual purchase orders (i.e. those that were not generated by a shipment) and summarizes them by the primary keys
# ========================================

spark.sql(f"""INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
(TransactionType, f_DataSource, f_FiscalMonthDateKey, f_PurchaseShipmentNumber, f_PurchaseShipmentLine,
 f_PurchaseReference, f_SaleReference, f_MillTicketDate, f_MillTicketNumber, f_SaleShipmentNumber,
 f_PurchaseNumber, f_PurchaseLine, f_SaleNumber, lkp_SALESTYPE, lkp_VENDORNUMBER, lkp_LEDGERVOUCHER, lkp_INVOICENUMBER, lkp_OFFICE)
SELECT 'Manual Purchase Order', CASE WHEN PD.Fiscal_Month = {CurrentFiscalMonth} THEN 'Summary' ELSE 'History' END,
       PD2.DateKey, 0, 0, COALESCE(POL.CUSTOMERREFERENCE, PO.PURCHASEORDERNUMBER, ''), '',
       CAST('1900-01-01' AS DATE), '', '0', '0', 0, 0,
       GL.SaleType, PO.ORDERVENDORACCOUNTNUMBER, PInvH.LEDGERVOUCHER, PInvH.INVOICENUMBER,
       SPLIT(CAST(PO.DEFAULTLEDGERDIMENSIONDISPLAYVALUE AS STRING), '-')[2]
FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_purchpurchaseorderheaderv2 PO
LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinepurchasekeys PKeys 
       ON PO.PURCHASEORDERNUMBER = PKeys.PURCHID AND PKeys.DATAAREAID = 'BRK' AND PKeys.IsDeleted = 0
LEFT OUTER JOIN (SELECT a.PURCHASEORDERNUMBER, MAX(a.CUSTOMERREFERENCE) AS CUSTOMERREFERENCE 
                 FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_purchpurchaseorderlinev2 a 
                 LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentlinepurchasekeys b 
                 ON a.PURCHASEORDERNUMBER = b.PURCHID AND b.DATAAREAID = 'BRK' AND b.IsDeleted = 0
                 WHERE a.DATAAREAID = 'BRK' AND a.IsDeleted = 0 AND COALESCE(a.CUSTOMERREFERENCE,'') <> '' AND b.PURCHID IS NULL
                 GROUP BY a.PURCHASEORDERNUMBER) POL 
       ON PO.PURCHASEORDERNUMBER = POL.PURCHASEORDERNUMBER
INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkpurchaseinvoiceheader PInvH 
       ON PO.PURCHASEORDERNUMBER = PInvH.PURCHASEORDERNUMBER AND PInvH.DATAAREAID = 'BRK' AND PInvH.IsDeleted = 0
INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date PD 
       ON CAST(PInvH.INVOICEDATE AS DATE) = CAST(PD.Date_Day AS DATE)
INNER JOIN {enriched_catalog}.{temp_schema}.temp_brk_fiscalmonthdates_d365 PD2 
       ON PD.Fiscal_Month = PD2.Fiscal_Month
LEFT OUTER JOIN (SELECT VOUCHER, DOCUMENTNUMBER, MAX(SaleType) AS SaleType FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 GROUP BY VOUCHER, DOCUMENTNUMBER) GL 
       ON PInvH.LEDGERVOUCHER COLLATE UTF8_LCASE = GL.VOUCHER AND PInvH.INVOICENUMBER COLLATE UTF8_LCASE = GL.DOCUMENTNUMBER
WHERE PO.IsDeleted = 0 AND PO.DATAAREAID = 'BRK' AND PKeys.PURCHID IS NULL
      AND (({MonthToLoad} = 999999 AND PD.Fiscal_Month = {CurrentFiscalMonth}) 
           OR ({MonthToLoad} = 888888 AND PD.Fiscal_Month IN ({CurrentFiscalMonth},{PriorFiscalMonth})) 
           OR ({MonthToLoad} = 777777 AND PD.Fiscal_Month = {PriorFiscalMonth}) 
           OR {MonthToLoad} = 111111 
           OR {MonthToLoad} = PD.Fiscal_Month)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# fourth section grabs manual journal entries not tied to shipments and summarizes them by the primary keys
# ========================================

spark.sql(f"""INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
(TransactionType, f_DataSource, f_FiscalMonthDateKey, f_PurchaseShipmentNumber, f_PurchaseShipmentLine,
 f_PurchaseReference, f_SaleReference, f_MillTicketDate, f_MillTicketNumber, f_SaleShipmentNumber,
 f_PurchaseNumber, f_PurchaseLine, f_SaleNumber, f_MillNetWgt, f_PurchasedWgt, f_SaleWgt,
 f_SaleAmount, f_Cost, lkp_SALESTYPE, lkp_VENDORNUMBER, lkp_MANUALCUSTOMERNUMBER, lkp_OFFICE)
SELECT GL.TransactionType, GL.DataSource, GL.FiscalMonthDateKey, 0, 0,
       CASE WHEN GL.TEXT = '' THEN GL.DESCRIPTION ELSE GL.TEXT END,
       CASE WHEN GL.TEXT = '' THEN GL.DESCRIPTION ELSE GL.TEXT END,
       CAST('1900-01-01' AS DATE), '', '0', '0', 0, 0,
       SUM(CAST(CASE WHEN GL.WEIGHT = '0' OR TRY_CAST(GL.WEIGHT AS FLOAT) IS NULL THEN NULL ELSE GL.WEIGHT END AS FLOAT)),
       SUM(CAST(CASE WHEN GL.WEIGHT = '0' OR TRY_CAST(GL.WEIGHT AS FLOAT) IS NULL THEN NULL ELSE GL.WEIGHT END AS FLOAT)),
       SUM(CAST(CASE WHEN GL.WEIGHT = '0' OR TRY_CAST(GL.WEIGHT AS FLOAT) IS NULL THEN NULL ELSE GL.WEIGHT END AS FLOAT)),
       SUM(GL.SaleAmount), SUM(GL.Cost), MAX(GL.SaleType), MAX(GL.VENDORNUMBER), MAX(GL.MANUALCUSTOMERNUMBER), MAX(GL.Office)
FROM {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 GL
LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoiceheaderv2 SInvH 
       ON GL.VOUCHER = SInvH.LEDGERVOUCHER COLLATE UTF8_LCASE AND SInvH.DATAAREAID = 'BRK' AND SInvH.IsDeleted = 0
LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkpurchaseinvoiceheader PInvH 
       ON GL.VOUCHER = PInvH.LEDGERVOUCHER COLLATE UTF8_LCASE AND PInvH.DATAAREAID = 'BRK' AND PInvH.IsDeleted = 0
WHERE GL.TransactionType = 'Manual Journal Entry' AND SInvH.LEDGERVOUCHER IS NULL AND PInvH.LEDGERVOUCHER IS NULL
GROUP BY GL.TransactionType, GL.DataSource, GL.FiscalMonthDateKey, CASE WHEN GL.TEXT = '' THEN GL.DESCRIPTION ELSE GL.TEXT END
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# fifth section grabs offset manual journal entries not tied to shipments and summarizes them by the primary keys
# ========================================

spark.sql(f"""INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
(TransactionType, f_DataSource, f_FiscalMonthDateKey, f_PurchaseShipmentNumber, f_PurchaseShipmentLine,
 f_PurchaseReference, f_SaleReference, f_MillTicketDate, f_MillTicketNumber, f_SaleShipmentNumber,
 f_PurchaseNumber, f_PurchaseLine, f_SaleNumber, f_MillNetWgt, f_PurchasedWgt, f_SaleWgt,
 f_SaleAmount, f_Cost, lkp_SALESTYPE, lkp_VENDORNUMBER, lkp_MANUALCUSTOMERNUMBER, lkp_OFFICE)
SELECT GL.TransactionType, GL.DataSource, GL.FiscalMonthDateKey, 0, 0,
       CASE WHEN GL.OFFSETTEXT = '' AND GL.TEXT = '' THEN GL.DESCRIPTION || ' Offset' 
            WHEN GL.OFFSETTEXT = '' AND GL.TEXT <> '' THEN GL.TEXT || ' Offset' 
            ELSE GL.OFFSETTEXT END,
       CASE WHEN GL.OFFSETTEXT = '' AND GL.TEXT = '' THEN GL.DESCRIPTION || ' Offset' 
            WHEN GL.OFFSETTEXT = '' AND GL.TEXT <> '' THEN GL.TEXT || ' Offset' 
            ELSE GL.OFFSETTEXT END,
       CAST('1900-01-01' AS DATE), '', '0', '0', 0, 0,
       SUM(CAST(CASE WHEN GL.WEIGHT = '0' OR TRY_CAST(GL.WEIGHT AS FLOAT) IS NULL THEN NULL ELSE GL.WEIGHT END AS FLOAT)),
       SUM(CAST(CASE WHEN GL.WEIGHT = '0' OR TRY_CAST(GL.WEIGHT AS FLOAT) IS NULL THEN NULL ELSE GL.WEIGHT END AS FLOAT)),
       SUM(CAST(CASE WHEN GL.WEIGHT = '0' OR TRY_CAST(GL.WEIGHT AS FLOAT) IS NULL THEN NULL ELSE GL.WEIGHT END AS FLOAT)),
       SUM(GL.SaleAmount), SUM(GL.Cost), MAX(GL.SaleType), MAX(GL.VENDORNUMBER), MAX(GL.MANUALCUSTOMERNUMBER), MAX(GL.Office)
FROM {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365 GL
LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_salesinvoiceheaderv2 SInvH 
       ON GL.VOUCHER = SInvH.LEDGERVOUCHER COLLATE UTF8_LCASE AND SInvH.DATAAREAID = 'BRK' AND SInvH.IsDeleted = 0
LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkpurchaseinvoiceheader PInvH 
       ON GL.VOUCHER = PInvH.LEDGERVOUCHER COLLATE UTF8_LCASE AND PInvH.DATAAREAID = 'BRK' AND PInvH.IsDeleted = 0
WHERE GL.TransactionType = 'Offset Manual Journal Entry' AND SInvH.LEDGERVOUCHER IS NULL AND PInvH.LEDGERVOUCHER IS NULL
GROUP BY GL.TransactionType, GL.DataSource, GL.FiscalMonthDateKey,
         CASE WHEN GL.OFFSETTEXT = '' AND GL.TEXT = '' THEN GL.DESCRIPTION || ' Offset' 
              WHEN GL.OFFSETTEXT = '' AND GL.TEXT <> '' THEN GL.TEXT || ' Offset' 
              ELSE GL.OFFSETTEXT END
""")

# ========================================
# VENDOR REVERSAL LOGIC
# Prior to adding vendor invoices, check for reversals. If the original entry and reversal are in the same fiscal month, remove both. If the original entry and reversal are in different fiscal months, leave it alone.
# ========================================

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW ReversalsVEND AS
SELECT DISTINCT
    a.VOUCHER AS Voucher,
    b.TraceNum AS TraceNum,
    d.Fiscal_Month AS FiscalMonth,
    d.Date_Day
FROM {bronze_catalog}.{SourceSchemaName}.{SourceTableName} a
INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brktransactionreversaltrans b
    ON a.GENERALJOURNALACCOUNTENTRYRECID = b.RefRecId
    AND a.LEDGERNAME = b.DataAreaId
    AND b.IsDeleted = 0
INNER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date d
    ON CAST(a.ACCOUNTINGDATE AS DATE) = d.Date_Day
INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkvendtrans VT
    ON a.VOUCHER = VT.VOUCHER
    AND VT.TRANSTYPE = 'VEND'
    AND VT.IsDeleted = 0
INNER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkpurchaseinvoiceheader V
    ON a.VOUCHER = V.LEDGERVOUCHER
    AND V.DATAAREAID = 'BRK'
    AND V.IsDeleted = 0
WHERE a.IsDeleted = 0
    AND a.LEDGERNAME = 'BRK'
    AND a.POSTINGTYPE = 'LedgerJournal'
    AND COALESCE(b.Reversed, '') = 'Yes'
    AND COALESCE(b.TraceNum, '') <> ''
""")


spark.sql(f"""
CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_genkeepvend_d365 AS
SELECT DISTINCT
    LJ.VOUCHER AS InitialVoucher,
    RV.Date_Day AS InitialDate,
    RVf.Date_Day AS ReverseDate,
    RVf.FiscalMonth
FROM {bronze_catalog}.{SourceSchemaName}.{SourceTableName} LJ
INNER JOIN ReversalsVEND RV
    ON LJ.VOUCHER = RV.Voucher
INNER JOIN ReversalsVEND RVf
    ON RV.TraceNum = RVf.TraceNum
    AND RV.Voucher <> RVf.Voucher
    AND RV.FiscalMonth <> RVf.FiscalMonth
""")


spark.sql(f"""
CREATE OR REPLACE TABLE {enriched_catalog}.{temp_schema}.temp_brk_genremovevend_d365 AS
SELECT DISTINCT
    LJ.VOUCHER AS InitialVoucher,
    RV.Date_Day AS InitialDate,
    RVf.Date_Day AS ReverseDate,
    LJ.JOURNALCATEGORY
FROM {bronze_catalog}.{SourceSchemaName}.{SourceTableName} LJ
INNER JOIN ReversalsVEND RV
    ON LJ.VOUCHER = RV.Voucher
INNER JOIN ReversalsVEND RVf
    ON RV.TraceNum = RVf.TraceNum
    AND RV.Voucher <> RVf.Voucher
    AND RV.FiscalMonth = RVf.FiscalMonth
""")


spark.sql(f"""
DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
WHERE VOUCHER IN (SELECT InitialVoucher COLLATE UTF8_LCASE FROM {enriched_catalog}.{temp_schema}.temp_brk_genremovevend_d365)
""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# sixth section grabs vendor invoice journals and summarizes them by the primary keys
# ========================================


spark.sql(f"""INSERT INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
(TransactionType, f_DataSource, f_FiscalMonthDateKey, f_PurchaseShipmentNumber, f_PurchaseShipmentLine,
 f_PurchaseReference, f_SaleReference, f_MillTicketDate, f_MillTicketNumber, f_SaleShipmentNumber,
 f_PurchaseNumber, f_PurchaseLine, f_SaleNumber, f_SaleAmount, f_Cost, lkp_SALESTYPE, lkp_VENDORNUMBER, lkp_MANUALCUSTOMERNUMBER, lkp_OFFICE, lkp_BUSINESSUNIT)
SELECT 'Vendor Invoice Journal', GL.DataSource, GL.FiscalMonthDateKey, 0, 0,
       COALESCE(VT.INVOICE,'') || ' ' || COALESCE(V.INVOICEVENDORACCOUNTNUMBER,''),
       COALESCE(VT.INVOICE,'') || ' ' || COALESCE(V.INVOICEVENDORACCOUNTNUMBER,''),
       CAST('1900-01-01' AS DATE), '', '0', '0', 0, 0,
       SUM(GL.SaleAmount), SUM(GL.Cost), MAX(GL.SaleType), MAX(V.INVOICEVENDORACCOUNTNUMBER), MAX(GL.DESCRIPTION), GL.Office, GL.BusinessUnit
FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365 GL
LEFT JOIN {enriched_catalog}.{temp_schema}.temp_brk_genkeepvend_d365 b
       ON GL.VOUCHER = b.InitialVoucher COLLATE UTF8_LCASE
       AND CAST(GL.ACCOUNTINGDATE AS DATE) = b.InitialDate
LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkvendtrans VT 
       ON GL.VOUCHER = VT.Voucher COLLATE UTF8_LCASE AND VT.TRANSTYPE = 'VEND' AND VT.IsDeleted = 0
LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkpurchaseinvoiceheader V 
       ON GL.VOUCHER = V.LEDGERVOUCHER COLLATE UTF8_LCASE AND V.IsDeleted = 0
WHERE GL.POSTINGTYPE = 'LedgerJournal' AND GL.JOURNALCATEGORY = 'VEND'
GROUP BY GL.DataSource, GL.FiscalMonthDateKey,
         COALESCE(VT.INVOICE,'') || ' ' || COALESCE(V.INVOICEVENDORACCOUNTNUMBER,''),
         GL.Office, GL.BusinessUnit
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# Set the OriginalID to -999999 since there is no unique ID for D365 transactions and Set the metadata - these are the same for all records in this ETL execution
# ========================================

spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
    SET f_OriginalID = -999999, 
        f_DJJGeneratedFlag = 0, 
        f_EnrichedTimestampUTC = current_timestamp()""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# Set the SourceSystemKey using the value pre-defined above (the foreign key that represents D365)
# ========================================

spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
    SET f_SourceSystemKey = COALESCE({SourceSystemKey}, -1)""")

spark.sql(f"""UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
    SET f_ServiceItem = '', f_ServiceItemKey = 0""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# Set the BuyPlanMonthKey to the sale contract buy period if it exists, otherwise the purchase contract buy period
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT f_PurchaseShipmentNumber, f_PurchaseShipmentLine, f_FiscalMonthDateKey, f_PurchaseReference, f_SaleReference, bpmk
    FROM (
        SELECT 
            P2.f_PurchaseShipmentNumber, 
            P2.f_PurchaseShipmentLine, 
            P2.f_FiscalMonthDateKey, 
            P2.f_PurchaseReference, 
            P2.f_SaleReference,
            COALESCE(SLD.Month, PLD.Month, 190001) AS bpmk,
            ROW_NUMBER() OVER (PARTITION BY P2.f_PurchaseShipmentNumber, P2.f_PurchaseShipmentLine, P2.f_FiscalMonthDateKey, P2.f_PurchaseReference, P2.f_SaleReference ORDER BY COALESCE(SLD.Month, PLD.Month, 190001) DESC) AS rn
        FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P2
        LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractheader PH
            ON P2.lkp_PURCHASECONTRACTID = PH.CONTRACTID
           AND PH.DATAAREAID = 'BRK'
           AND PH.IsDeleted = 0
        LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractline PL
            ON P2.lkp_PURCHASECONTRACTID = PL.CONTRACTID
           AND P2.lkp_PURCHASELINE = PL.CONTRACTLINE
           AND PL.DATAAREAID = 'BRK'
           AND PL.IsDeleted = 0
        LEFT OUTER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date PLD
            ON PL.BUYPERIOD = CAST(PLD.Date_Day AS DATE)
        LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractheader SLH
            ON P2.lkp_SALESCONTRACTID = SLH.CONTRACTID
           AND SLH.DATAAREAID = 'BRK'
           AND SLH.IsDeleted = 0
        LEFT OUTER JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractline SLL
            ON P2.lkp_SALESCONTRACTID = SLL.CONTRACTID
           AND P2.lkp_SALESLINE = SLL.CONTRACTLINE
           AND SLL.DATAAREAID = 'BRK'
           AND SLL.IsDeleted = 0
        LEFT OUTER JOIN {enriched_catalog}.{SourceSchemaName}.dim_date SLD
            ON SLL.BUYPERIOD = CAST(SLD.Date_Day AS DATE)
    ) ranked
    WHERE rn = 1
) src
ON P.f_PurchaseShipmentNumber = src.f_PurchaseShipmentNumber
   AND P.f_PurchaseShipmentLine = src.f_PurchaseShipmentLine
   AND P.f_FiscalMonthDateKey = src.f_FiscalMonthDateKey
   AND P.f_PurchaseReference = src.f_PurchaseReference
   AND P.f_SaleReference = src.f_SaleReference
WHEN MATCHED THEN 
  UPDATE SET P.f_BuyPlanMonthKey = src.bpmk
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# Set the BrokerageGroupKey using the value pre-defined above (the foreign key that represents Ferrous Brokerage)
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_BrokerageGroupKey = COALESCE({BrokerageGroupKey}, -1)
""")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING {enriched_catalog}.{DestinationSchemaName}.dim_saletype st
ON P.lkp_SALESTYPE = st.SaleTypeCode COLLATE UTF8_LCASE
WHEN MATCHED THEN
  UPDATE SET P.f_SaleTypeKey = COALESCE(st.SaleTypeKey, -1)
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET P.f_SaleTypeKey = -1
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# Update grades for Shipments
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT 
        P2.f_PurchaseShipmentNumber,
        P2.f_PurchaseShipmentLine,
        P2.f_FiscalMonthDateKey,
        COALESCE(G2.GradeKey, G.GradeKey, -1) AS GradeKey
    FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P2
    LEFT JOIN {enriched_catalog}.{DestinationSchemaName}.dim_grade G
        ON P2.lkp_ITEMCODE = G.GradeCode COLLATE UTF8_LCASE
        AND G.CompanyCode = 'B'
    LEFT JOIN (
        SELECT GradeCode, MIN(GradeKey) AS GradeKey
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_grade
        GROUP BY GradeCode
    ) G2
        ON P2.lkp_ITEMCODE = G2.GradeCode COLLATE UTF8_LCASE
        AND G.GradeKey IS NULL
    WHERE P2.TransactionType = 'Shipment'
) src
ON P.f_PurchaseShipmentNumber = src.f_PurchaseShipmentNumber
   AND P.f_PurchaseShipmentLine = src.f_PurchaseShipmentLine
   AND P.f_FiscalMonthDateKey = src.f_FiscalMonthDateKey
WHEN MATCHED THEN
  UPDATE SET 
      P.f_PurchaseGradeKey = src.GradeKey,
      P.f_SaleGradeKey = src.GradeKey
""")


# ========================================
# Set grades to 0 for non-shipments. Grades do not apply for non-shipments
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_PurchaseGradeKey = 0,
    f_SaleGradeKey = 0
WHERE TransactionType <> 'Shipment'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT P2.f_PurchaseShipmentNumber, P2.f_PurchaseShipmentLine, P2.f_FiscalMonthDateKey,
           COALESCE(VT.VehicleTypeKey,-1) AS VehicleTypeKey
    FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P2
    LEFT JOIN {enriched_catalog}.{SourceSchemaName}.dim_vehicletype VT
        ON P2.lkp_MODE = VT.VehicleTypeDescription COLLATE UTF8_LCASE
    WHERE P2.TransactionType = 'Shipment'
) src
ON P.f_PurchaseShipmentNumber = src.f_PurchaseShipmentNumber
   AND P.f_PurchaseShipmentLine = src.f_PurchaseShipmentLine
   AND P.f_FiscalMonthDateKey = src.f_FiscalMonthDateKey
WHEN MATCHED THEN UPDATE SET P.f_VehicleTypeKey = src.VehicleTypeKey
""")


# ========================================
# Vehicle types do not apply for non-shipments
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_VehicleTypeKey = 0
WHERE TransactionType <> 'Shipment'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.AddressBookID, sub.AddressBookKey
    FROM (
        SELECT ab.AddressBookID, ab.AddressBookKey,
               ROW_NUMBER() OVER (PARTITION BY ab.AddressBookID ORDER BY ab.AddressBookKey) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_addressbook ab
    ) sub
    WHERE sub.rn = 1
) src
ON P.lkp_VENDORNUMBER = src.AddressBookID COLLATE UTF8_LCASE
WHEN MATCHED AND P.TransactionType <> 'Manual Sale Order' THEN 
    UPDATE SET P.f_SupplierAddressBookKey = src.AddressBookKey
WHEN NOT MATCHED BY SOURCE AND P.TransactionType <> 'Manual Sale Order' THEN 
    UPDATE SET P.f_SupplierAddressBookKey = -1
""")



# ========================================
# Suppliers do not apply for manual sale orders
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_SupplierAddressBookKey = 0
WHERE TransactionType = 'Manual Sale Order'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT P2.f_PurchaseShipmentNumber, P2.f_PurchaseShipmentLine, P2.f_FiscalMonthDateKey,
           COALESCE(OL.LocationKey,-1) AS OriginLocationKey
    FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P2
    LEFT JOIN {enriched_catalog}.{DestinationSchemaName}.dim_location OL
        ON P2.lkp_ORIGIN = OL.LocationID
    WHERE P2.TransactionType = 'Shipment'
) src
ON P.f_PurchaseShipmentNumber = src.f_PurchaseShipmentNumber
   AND P.f_PurchaseShipmentLine = src.f_PurchaseShipmentLine
   AND P.f_FiscalMonthDateKey = src.f_FiscalMonthDateKey
WHEN MATCHED THEN UPDATE SET P.f_OriginLocationKey = src.OriginLocationKey
""")



# ========================================
# Origin locations do not apply for non-shipments
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_OriginLocationKey = 0
WHERE TransactionType <> 'Shipment'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# Set the ConsumerKey for shipments, manual sale orders, manual journal entries, offset manual journal entries, and vendor invoice journals
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.ConsumerID, sub.ConsumerKey
    FROM (
        SELECT c.ConsumerID, c.ConsumerKey,
               ROW_NUMBER() OVER (PARTITION BY c.ConsumerID ORDER BY c.ConsumerKey) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_consumer c
    ) sub
    WHERE sub.rn = 1
) src
ON LEFT(P.lkp_MANUALCUSTOMERNUMBER, 20) = src.ConsumerID COLLATE UTF8_LCASE
WHEN MATCHED AND P.TransactionType NOT IN ('Manual Purchase Order') THEN 
    UPDATE SET P.f_ConsumerKey = src.ConsumerKey
WHEN NOT MATCHED BY SOURCE AND P.TransactionType NOT IN ('Manual Purchase Order','Vendor Invoice Journal') THEN 
    UPDATE SET P.f_ConsumerKey = -1
WHEN NOT MATCHED BY SOURCE AND P.TransactionType = 'Vendor Invoice Journal' THEN 
    UPDATE SET P.f_ConsumerKey = 0
""")


# ========================================
# Consumers do not apply for manual purchase orders
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_ConsumerKey = 0
WHERE TransactionType = 'Manual Purchase Order'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# For shipments, this is freight location where the material is being shipped to
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.LocationID, sub.LocationKey
    FROM (
        SELECT dl.LocationID, dl.LocationKey,
               ROW_NUMBER() OVER (PARTITION BY dl.LocationID ORDER BY dl.LocationKey) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_location dl
    ) sub
    WHERE sub.rn = 1
) src
ON P.lkp_DESTINATION = src.LocationID
WHEN MATCHED AND P.TransactionType = 'Shipment' THEN 
    UPDATE SET P.f_DestinationLocationKey = COALESCE(src.LocationKey, -1)
WHEN NOT MATCHED BY SOURCE AND P.TransactionType = 'Shipment' THEN 
    UPDATE SET P.f_DestinationLocationKey = -1
""")


# ========================================
# For non-shipments, this is the consumer's physical address
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.ConsumerKey, sub.LocationKey
    FROM (
        SELECT c.ConsumerKey, COALESCE(dl.LocationKey, dl2.LocationKey) AS LocationKey,
               ROW_NUMBER() OVER (PARTITION BY c.ConsumerKey ORDER BY COALESCE(dl.LocationKey, dl2.LocationKey)) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_consumer c
        LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_custcustomerv3 c2
            ON c.ConsumerID = c2.CUSTOMERACCOUNT
            AND c2.DATAAREAID = 'BRK'
            AND c2.IsDeleted = 0
        LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_dirpartylocationpostaladdressv2 d
            ON c2.PARTYNUMBER = d.PARTYNUMBER
            AND d.IsDeleted = 0
            AND d.ROLES LIKE '%PHYSICAL%'
        LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brklogisticspostaladdress e
            ON d.LOCATIONID = e.LOCATIONID
            AND e.IsDeleted = 0
        LEFT JOIN {enriched_catalog}.{DestinationSchemaName}.dim_location dl
            ON d.LOCATIONID = dl.LocationID
        LEFT JOIN {enriched_catalog}.{DestinationSchemaName}.dim_location dl2
            ON e.LOCATION = dl2.LocationID
            AND dl2.MasterSourceSystem = 'MS Dynamics 365'
            AND dl.LocationID IS NULL
    ) sub
    WHERE sub.rn = 1
) src
ON P.f_ConsumerKey = src.ConsumerKey
WHEN MATCHED AND P.TransactionType <> 'Shipment' THEN 
    UPDATE SET P.f_DestinationLocationKey = COALESCE(src.LocationKey, -1)
WHEN NOT MATCHED BY SOURCE AND P.TransactionType <> 'Shipment' THEN 
    UPDATE SET P.f_DestinationLocationKey = -1
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# For shipments, this is office from the purchase contract that the shipment was generated from.if the office cannot be found, set this to the Unknown office under the Ferrous Brokerage group
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT P2.f_PurchaseShipmentNumber, P2.f_PurchaseShipmentLine, P2.f_FiscalMonthDateKey,
           COALESCE(PBO.BrokerageOfficeKey, {DefaultOfficeKey}) AS PurchaseOfficeKey
    FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P2
    LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractheader PH
        ON P2.lkp_PURCHASECONTRACTID = PH.CONTRACTID
        AND PH.DATAAREAID = 'BRK'
        AND PH.IsDeleted = 0
    LEFT JOIN {enriched_catalog}.{DestinationSchemaName}.dim_brokerageoffice PBO
        ON PH.VENDOROFFICE = PBO.AccountNumber
        AND PBO.BrokerageGroup = 'Ferrous Brokerage'
    WHERE P2.TransactionType = 'Shipment'
) src
ON P.f_PurchaseShipmentNumber = src.f_PurchaseShipmentNumber
   AND P.f_PurchaseShipmentLine = src.f_PurchaseShipmentLine
   AND P.f_FiscalMonthDateKey = src.f_FiscalMonthDateKey
WHEN MATCHED THEN UPDATE SET P.f_PurchaseOfficeKey = src.PurchaseOfficeKey
""")


spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_PurchaseOfficeKey = {DefaultOfficeKey}
WHERE TransactionType NOT IN ('Shipment','Manual Sale Order')
""")


spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.AccountNumber, sub.BrokerageOfficeKey
    FROM (
        SELECT bo.AccountNumber, bo.BrokerageOfficeKey,
               ROW_NUMBER() OVER (PARTITION BY bo.AccountNumber ORDER BY bo.BrokerageOfficeKey) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_brokerageoffice bo
    ) sub
    WHERE sub.rn = 1
) src
ON P.lkp_OFFICE = src.AccountNumber COLLATE UTF8_LCASE
WHEN MATCHED AND P.TransactionType NOT IN ('Shipment','Manual Sale Order') THEN
    UPDATE SET P.f_PurchaseOfficeKey = src.BrokerageOfficeKey
""")

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.AddressBookID, sub.CurrentBrokerageOfficeKey
    FROM (
        SELECT ab.AddressBookID, ab.CurrentBrokerageOfficeKey,
               ROW_NUMBER() OVER (PARTITION BY ab.AddressBookID ORDER BY ab.CurrentBrokerageOfficeKey) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_addressbook ab
    ) sub
    WHERE sub.rn = 1
) src
ON P.lkp_VENDORNUMBER = src.AddressBookID COLLATE UTF8_LCASE
WHEN MATCHED AND P.TransactionType NOT IN ('Shipment','Manual Sale Order') THEN
    UPDATE SET P.f_PurchaseOfficeKey = src.CurrentBrokerageOfficeKey
""")


# ========================================
# Purchase office does not apply for manual sale orders
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_PurchaseOfficeKey = 0
WHERE TransactionType = 'Manual Sale Order'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# For shipments, this is customer office from the sale contract that the shipment was generated from. If there is no sale contract, choose the customer office from the purchase contract. If the office cannot be found on either, set this to the Unknown office under the Ferrous Brokerage group
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT P2.f_PurchaseShipmentNumber, P2.f_PurchaseShipmentLine, P2.f_FiscalMonthDateKey,
           COALESCE(SBO.BrokerageOfficeKey, PBO.BrokerageOfficeKey, {DefaultOfficeKey}) AS SaleOfficeKey
    FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P2
    LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractheader SH
        ON P2.lkp_SALESCONTRACTID = SH.CONTRACTID
        AND SH.DATAAREAID = 'BRK'
        AND SH.IsDeleted = 0
    LEFT JOIN {enriched_catalog}.{DestinationSchemaName}.dim_brokerageoffice SBO
        ON SH.CUSTOMEROFFICE = SBO.AccountNumber
        AND SBO.BrokerageGroup = 'Ferrous Brokerage'
    LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractheader PH
        ON P2.lkp_PURCHASECONTRACTID = PH.CONTRACTID
        AND PH.DATAAREAID = 'BRK'
        AND PH.IsDeleted = 0
    LEFT JOIN {enriched_catalog}.{DestinationSchemaName}.dim_brokerageoffice PBO
        ON PH.CUSTOMEROFFICE = PBO.AccountNumber
        AND PBO.BrokerageGroup = 'Ferrous Brokerage'
    WHERE P2.TransactionType = 'Shipment'
) src
ON P.f_PurchaseShipmentNumber = src.f_PurchaseShipmentNumber
   AND P.f_PurchaseShipmentLine = src.f_PurchaseShipmentLine
   AND P.f_FiscalMonthDateKey = src.f_FiscalMonthDateKey
WHEN MATCHED THEN UPDATE SET P.f_SaleOfficeKey = src.SaleOfficeKey
""")



spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_SaleOfficeKey = {DefaultOfficeKey}
WHERE TransactionType NOT IN ('Shipment','Vendor Invoice Journal')
""")


spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.AccountNumber, sub.BrokerageOfficeKey
    FROM (
        SELECT fd.AccountNumber, fd.BrokerageOfficeKey,
               ROW_NUMBER() OVER (PARTITION BY fd.AccountNumber ORDER BY fd.BrokerageOfficeKey) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_brokerageoffice fd
    ) sub
    WHERE sub.rn = 1
) src
ON P.lkp_OFFICE = src.AccountNumber COLLATE UTF8_LCASE
WHEN MATCHED AND P.TransactionType NOT IN ('Shipment','Vendor Invoice Journal') THEN
    UPDATE SET P.f_SaleOfficeKey = src.BrokerageOfficeKey
""")


spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.AddressBookID, sub.CurrentBrokerageOfficeKey
    FROM (
        SELECT ab.AddressBookID, ab.CurrentBrokerageOfficeKey,
               ROW_NUMBER() OVER (PARTITION BY ab.AddressBookID ORDER BY ab.CurrentBrokerageOfficeKey) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_addressbook ab
    ) sub
    WHERE sub.rn = 1
) src
ON P.lkp_MANUALCUSTOMERNUMBER = src.AddressBookID COLLATE UTF8_LCASE
WHEN MATCHED AND P.TransactionType NOT IN ('Shipment','Vendor Invoice Journal') THEN
    UPDATE SET P.f_SaleOfficeKey = src.CurrentBrokerageOfficeKey
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING {enriched_catalog}.{DestinationSchemaName}.dim_consumer con
ON P.lkp_MANUALCUSTOMERNUMBER = con.ConsumerID COLLATE UTF8_LCASE
   AND P.TransactionType = 'Vendor Invoice Journal'
WHEN MATCHED THEN 
  UPDATE SET P.lkp_OFFICE = con.CurrentBrokerageOfficeName
""")


spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.BrokerageGroupKey, sub.BrokerageOfficeKey
    FROM (
        SELECT bo2.BrokerageGroupKey, bo2.BrokerageOfficeKey,
               ROW_NUMBER() OVER (PARTITION BY bo2.BrokerageGroupKey ORDER BY bo2.BrokerageOfficeKey) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_brokerageoffice bo2
        WHERE bo2.Office = 'Unknown'
    ) sub
    WHERE sub.rn = 1
) src
ON P.f_BrokerageGroupKey = src.BrokerageGroupKey
WHEN MATCHED AND P.TransactionType = 'Vendor Invoice Journal' THEN
    UPDATE SET P.f_SaleOfficeKey = src.BrokerageOfficeKey
""")


spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.AccountNumber, sub.BrokerageGroupKey, sub.BrokerageOfficeKey
    FROM (
        SELECT bo1.AccountNumber, bo1.BrokerageGroupKey, bo1.BrokerageOfficeKey,
               ROW_NUMBER() OVER (PARTITION BY bo1.AccountNumber, bo1.BrokerageGroupKey ORDER BY bo1.BrokerageOfficeKey) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_brokerageoffice bo1
    ) sub
    WHERE sub.rn = 1
) src
ON P.lkp_OFFICE = src.AccountNumber COLLATE UTF8_LCASE
   AND P.f_BrokerageGroupKey = src.BrokerageGroupKey
WHEN MATCHED AND P.TransactionType = 'Vendor Invoice Journal' THEN
    UPDATE SET P.f_SaleOfficeKey = src.BrokerageOfficeKey
""")


spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.ConsumerID, sub.CurrentBrokerageOfficeKey
    FROM (
        SELECT con1.ConsumerID, con1.CurrentBrokerageOfficeKey,
               ROW_NUMBER() OVER (PARTITION BY con1.ConsumerID ORDER BY con1.CurrentBrokerageOfficeKey) AS rn
        FROM {enriched_catalog}.{DestinationSchemaName}.dim_consumer con1
    ) sub
    WHERE sub.rn = 1
) src
ON P.lkp_MANUALCUSTOMERNUMBER = src.ConsumerID COLLATE UTF8_LCASE
WHEN MATCHED AND P.TransactionType = 'Vendor Invoice Journal' THEN
    UPDATE SET P.f_SaleOfficeKey = src.CurrentBrokerageOfficeKey
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_BOLNumber = ''
WHERE f_BOLNumber IS NULL
""")

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_ProfitPropertiesKey = {ProfitPropertiesKey}
""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# Non-shipments get the default value
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_ShipmentPropertiesKey = {DefaultShipmentPropertiesKey}
WHERE TransactionType <> 'Shipment'
""")

# ========================================
# Shipments get the value from the dimShipmentProperties lookup
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT P2.f_PurchaseShipmentNumber, P2.f_PurchaseShipmentLine, P2.f_FiscalMonthDateKey,
           COALESCE(prop.ShipmentPropertiesKey, {DefaultShipmentPropertiesKey}) AS ShipmentPropertiesKey
    FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P2
    LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentheader SH
        ON P2.f_PurchaseShipmentNumber = SH.SHIPMENTID
        AND SH.DATAAREAID = 'BRK'
        AND SH.IsDeleted = 0
    LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkshipmentline SL
        ON SH.SHIPMENTID = SL.SHIPMENTID
        AND SL.BRKSHIPMENTLINE = 1
        AND SL.DATAAREAID = 'BRK'
        AND SL.IsDeleted = 0
    LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractheader PH
        ON SL.PURCHASECONTRACTID = PH.CONTRACTID
        AND PH.DATAAREAID = 'BRK'
        AND PH.IsDeleted = 0
    LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractline PL
        ON SL.PURCHASECONTRACTID = PL.CONTRACTID
        AND SL.PURCHASELINE = PL.CONTRACTLINE
        AND PL.DATAAREAID = 'BRK'
        AND PL.IsDeleted = 0
    LEFT JOIN {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractheader SOH
        ON SL.SALESCONTRACTID = SOH.CONTRACTID
        AND SOH.DATAAREAID = 'BRK'
        AND SOH.IsDeleted = 0
    LEFT JOIN {bronze_catalog}.djj_brk.frt_sys_rate R
        ON SL.RATEID = R.RateID
        AND R.IsDeleted = 0
    LEFT JOIN {enriched_catalog}.{DestinationSchemaName}.dim_shipmentproperties prop
        ON CASE 
               WHEN COALESCE(SL.BRKFREIGHTRESPONSIBILITY,'') = 'DJJ' THEN 'DJJ'
               WHEN COALESCE(SL.BRKFREIGHTRESPONSIBILITY,'') = 'Vendor' THEN 'SHP'
               WHEN COALESCE(SL.BRKFREIGHTRESPONSIBILITY,'') = 'Customer' THEN 'CON-FRT'
               WHEN COALESCE(SL.BRKFREIGHTRESPONSIBILITY,'') = 'NoFreight' THEN 'NOF'
               ELSE ''
           END = prop.FreightResponsibilityCode
        AND prop.PaymentStatusCode = 'U'
        AND prop.ShipmentStatus = COALESCE(SH.SHIPMENTSTATUS, 'Unknown')
        AND CASE 
               WHEN COALESCE(PH.ORDERTYPE,'') = 'AGENCY' OR COALESCE(SOH.ORDERTYPE,'') = 'AGENCY' THEN 'A'
               WHEN COALESCE(PH.ORDERTYPE,'') = 'OPEN MKT' OR COALESCE(SOH.ORDERTYPE,'') = 'OPEN MKT' THEN 'O'
               ELSE 'U'
           END = prop.ContractTypeCode
        AND prop.MultiLegCode = COALESCE(CAST(R.MultiLeg AS INT), -1)
        AND CASE 
               WHEN COALESCE(SH.RejectionStatus,'') IN ('RejectionStarted','OutrightRejection','PartialRejection') THEN 'Rejected'
               ELSE 'Not Rejected'
           END = prop.Rejected
    WHERE P2.TransactionType = 'Shipment'
) src
ON P.f_PurchaseShipmentNumber = src.f_PurchaseShipmentNumber
   AND P.f_PurchaseShipmentLine = src.f_PurchaseShipmentLine
   AND P.f_FiscalMonthDateKey = src.f_FiscalMonthDateKey
WHEN MATCHED THEN UPDATE SET P.f_ShipmentPropertiesKey = src.ShipmentPropertiesKey
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.CONTRACTID, sub.EFFECTIVEDATE
    FROM (
        SELECT PH.CONTRACTID, CAST(PH.EFFECTIVEDATE AS DATE) AS EFFECTIVEDATE,
               ROW_NUMBER() OVER (PARTITION BY PH.CONTRACTID ORDER BY PH.EFFECTIVEDATE) AS rn
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractheader PH
        WHERE PH.DATAAREAID = 'BRK'
          AND PH.IsDeleted = 0
    ) sub
    WHERE sub.rn = 1
) src
ON P.lkp_PURCHASECONTRACTID = src.CONTRACTID
WHEN MATCHED THEN 
    UPDATE SET P.f_PurchaseContractDate = src.EFFECTIVEDATE
WHEN NOT MATCHED BY SOURCE THEN 
    UPDATE SET P.f_PurchaseContractDate = CAST('1900-01-01' AS DATE)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT sub.CONTRACTID, sub.EFFECTIVEDATE
    FROM (
        SELECT SLH.CONTRACTID, CAST(SLH.EFFECTIVEDATE AS DATE) AS EFFECTIVEDATE,
               ROW_NUMBER() OVER (PARTITION BY SLH.CONTRACTID ORDER BY SLH.EFFECTIVEDATE) AS rn
        FROM {bronze_catalog}.{SourceSchemaName}.d365fo_ods_d365fo_brkcontractheader SLH
        WHERE SLH.DATAAREAID = 'BRK'
          AND SLH.IsDeleted = 0
    ) sub
    WHERE sub.rn = 1
) src
ON P.lkp_SALESCONTRACTID = src.CONTRACTID
WHEN MATCHED THEN 
    UPDATE SET P.f_SaleEffectiveDate = src.EFFECTIVEDATE
WHEN NOT MATCHED BY SOURCE THEN 
    UPDATE SET P.f_SaleEffectiveDate = CAST('1900-01-01' AS DATE)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ========================================
# Set MillNetWgt for shipments by grabbing the quantity on the invoice.
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT P2.f_PurchaseShipmentNumber, P2.f_PurchaseShipmentLine, P2.f_FiscalMonthDateKey,
           COALESCE(PInvL.INVOICEDQUANTITY, SInvL.INVOICEDQUANTITY) AS MillNetWgt
    FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P2
    LEFT JOIN {enriched_catalog}.{temp_schema}.temp_brk_purchinvline_d365 PInvL
        ON P2.f_FiscalMonthDateKey = PInvL.FiscalMonthDateKey
        AND P2.f_PurchaseShipmentNumber = PInvL.SHIPMENTID
        AND P2.f_PurchaseShipmentLine = 1
    LEFT JOIN {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365 SInvL
        ON P2.f_FiscalMonthDateKey = SInvL.FiscalMonthDateKey
        AND P2.f_PurchaseShipmentNumber = SInvL.SHIPMENTID
        AND P2.f_PurchaseShipmentLine = 1
    WHERE P2.TransactionType = 'Shipment'
) src
ON P.f_PurchaseShipmentNumber = src.f_PurchaseShipmentNumber
   AND P.f_PurchaseShipmentLine = src.f_PurchaseShipmentLine
   AND P.f_FiscalMonthDateKey = src.f_FiscalMonthDateKey
WHEN MATCHED THEN UPDATE SET P.f_MillNetWgt = src.MillNetWgt
""")


# ========================================
# MillNetWgt does not apply to manual purchase orders, manual sale orders, or vendor invoice journals
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_MillNetWgt = NULL
WHERE TransactionType IN ('Manual Sale Order','Manual Purchase Order','Vendor Invoice Journal')
""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# Set PurchasedWgt for shipments by grabbing the quantity on the invoice
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT P2.f_PurchaseShipmentNumber, P2.f_PurchaseShipmentLine, P2.f_FiscalMonthDateKey,
           COALESCE(PInvL.INVOICEDQUANTITY, SInvL.INVOICEDQUANTITY) AS PurchasedWgt
    FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P2
    LEFT JOIN {enriched_catalog}.{temp_schema}.temp_brk_purchinvline_d365 PInvL
        ON P2.f_FiscalMonthDateKey = PInvL.FiscalMonthDateKey
        AND P2.f_PurchaseShipmentNumber = PInvL.SHIPMENTID
        AND P2.f_PurchaseShipmentLine = 1
    LEFT JOIN {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365 SInvL
        ON P2.f_FiscalMonthDateKey = SInvL.FiscalMonthDateKey
        AND P2.f_PurchaseShipmentNumber = SInvL.SHIPMENTID
        AND P2.f_PurchaseShipmentLine = 1
    WHERE P2.TransactionType = 'Shipment'
) src
ON P.f_PurchaseShipmentNumber = src.f_PurchaseShipmentNumber
   AND P.f_PurchaseShipmentLine = src.f_PurchaseShipmentLine
   AND P.f_FiscalMonthDateKey = src.f_FiscalMonthDateKey
WHEN MATCHED THEN UPDATE SET P.f_PurchasedWgt = src.PurchasedWgt
""")


# ========================================
# PurchasedWgt does not apply to manual purchase orders, manual sale orders, or vendor invoice journals. PurchasedWgt for manual journal entries and offset manual journal entries is set during the initial temp table creation
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_PurchasedWgt = NULL
WHERE TransactionType IN ('Manual Sale Order','Manual Purchase Order','Vendor Invoice Journal')
""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# Set SaleWgt for shipments by grabbing the quantity on the invoice
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
USING (
    SELECT P2.f_PurchaseShipmentNumber, P2.f_PurchaseShipmentLine, P2.f_FiscalMonthDateKey,
           SInvL.INVOICEDQUANTITY AS SaleWgt
    FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P2
    LEFT JOIN {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365 SInvL
        ON P2.f_FiscalMonthDateKey = SInvL.FiscalMonthDateKey
        AND P2.f_PurchaseShipmentNumber = SInvL.SHIPMENTID
        AND P2.f_PurchaseShipmentLine = 1
    WHERE P2.TransactionType = 'Shipment'
) src
ON P.f_PurchaseShipmentNumber = src.f_PurchaseShipmentNumber
   AND P.f_PurchaseShipmentLine = src.f_PurchaseShipmentLine
   AND P.f_FiscalMonthDateKey = src.f_FiscalMonthDateKey
WHEN MATCHED THEN UPDATE SET P.f_SaleWgt = src.SaleWgt
""")



# ========================================
# SaleWgt does not apply to manual purchase orders or manual sale orders, or vendor invoice journals. SaleWgt for manual journal entries and offset manual journal entries is set during the initial temp table creation
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_SaleWgt = NULL
WHERE TransactionType IN ('Manual Sale Order','Manual Purchase Order','Vendor Invoice Journal')
""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# SaleAmount for shipments, manual journal entries, offset manual journal entries, and vendor invoice journals is set during the initial temp table creation.
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
SET P.f_SaleAmount = (
    SELECT SUM(TRANSACTIONCURRENCYAMOUNT) * -1
    FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
    WHERE GLAccountFirstDigit = '4'
      AND VOUCHER = P.lkp_LEDGERVOUCHER
      AND DOCUMENTNUMBER = P.lkp_INVOICENUMBER
)
WHERE P.TransactionType = 'Manual Purchase Order'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# Cost for shipments, manual journal entries, offset manual journal entries, and vendor invoice journals is set during the initial temp table creation
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365 P
SET P.f_Cost = (
    SELECT SUM(TRANSACTIONCURRENCYAMOUNT) * -1
    FROM {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365
    WHERE GLAccountFirstDigit = '5'
      AND VOUCHER = P.lkp_LEDGERVOUCHER
      AND DOCUMENTNUMBER = P.lkp_INVOICENUMBER
)
WHERE P.TransactionType = 'Manual Purchase Order'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
# ========================================
# The difference between Sales and Cost
# ========================================

spark.sql(f"""
UPDATE {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
SET f_Margin = CASE 
                 WHEN f_SaleAmount IS NULL AND f_Cost IS NULL THEN NULL
                 ELSE COALESCE(f_SaleAmount,0) - COALESCE(f_Cost,0)
               END
""")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql(f"""
DELETE FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365
WHERE COALESCE(f_SaleAmount,0) = 0 AND COALESCE(f_Cost,0) = 0
""")

DataFrame[num_affected_rows: bigint]

# 4. TABLE LOAD (UPDATE / INSERT / DELETE)

In [0]:
# ========================================
# Remove the data for the month being run from the fact table so that it can be reloaded. This is because the DataSource column is part of the primary key, yet it changes with the fiscal month
# ========================================

spark.sql(f"""
MERGE INTO {enriched_catalog}.{DestinationSchemaName}.{DestinationTableName} a
USING {enriched_catalog}.{SourceSchemaName}.dim_date d
ON a.FiscalMonthDateKey = d.DateKey
WHEN MATCHED AND a.SourceSystemKey = 3
     AND (
         ({MonthToLoad} = 999999 AND d.Fiscal_Month = {CurrentFiscalMonth})
         OR ({MonthToLoad} = 888888 AND d.Fiscal_Month IN ({CurrentFiscalMonth},{PriorFiscalMonth}))
         OR ({MonthToLoad} = 777777 AND d.Fiscal_Month = {PriorFiscalMonth})
         OR {MonthToLoad} = 111111
         OR {MonthToLoad} = d.Fiscal_Month
     )
THEN DELETE
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
merge_history = spark.sql(f"DESCRIBE HISTORY {enriched_catalog}.{DestinationSchemaName}.{DestinationTableName}").filter("operation = 'MERGE'").first()
if merge_history is not None:
    metrics = merge_history["operationMetrics"]
    RowsDeleted = int(metrics.get("numTargetRowsDeleted", 0))
else:
    RowsDeleted = 0

print(f"Rows deleted: {RowsDeleted}")

Rows deleted: 112427


In [0]:
# ============================================================
# INSERT RECORDS
# ============================================================

spark.sql(f"""INSERT INTO {enriched_catalog}.{DestinationSchemaName}.{DestinationTableName} (	 
            OriginalID,
		    DataSource,
		    SourceSystemKey,
		    FiscalMonthDateKey,
		    PurchaseShipmentNumber,
		    PurchaseShipmentLine,
		    PurchaseReference,
		    SaleReference,
		    ServiceItem,
		    BuyPlanMonthKey,
		    BrokerageGroupKey,
		    SaleTypeKey,
		    PurchaseGradeKey,
		    SaleGradeKey,
		    ServiceItemKey,
		    VehicleTypeKey,
		    SupplierAddressBookKey,
		    OriginLocationKey,
		    ConsumerKey,
		    DestinationLocationKey,
		    PurchaseOfficeKey,
		    SaleOfficeKey,
		    ProfitPropertiesKey,
            ShipmentPropertiesKey,
            OriginTicketNumber,
		    MillTicketDate,
		    MillTicketNumber,
		    PurchaseContractDate,
		    SaleEffectiveDate,
		    SaleShipmentNumber,
		    PurchaseNumber,
		    PurchaseLine,
		    SaleNumber,
            BOLNumber,
		    MillNetWgt,
		    PurchasedWgt,
		    SaleWgt,
		    SaleAmount,
		    Margin,
		    Cost,
		    DJJGeneratedFlag,
		    EnrichedTimestampUTC
        )
    SELECT f_OriginalID, f_DataSource, f_SourceSystemKey, f_FiscalMonthDateKey, f_PurchaseShipmentNumber, f_PurchaseShipmentLine,
        f_PurchaseReference, f_SaleReference, f_ServiceItem, f_BuyPlanMonthKey, f_BrokerageGroupKey,
        f_SaleTypeKey, f_PurchaseGradeKey, f_SaleGradeKey, f_ServiceItemKey, f_VehicleTypeKey,
        f_SupplierAddressBookKey, f_OriginLocationKey, f_ConsumerKey, f_DestinationLocationKey,
        f_PurchaseOfficeKey, f_SaleOfficeKey, f_ProfitPropertiesKey, f_ShipmentPropertiesKey,
        f_OriginTicketNumber, f_MillTicketDate, f_MillTicketNumber, f_PurchaseContractDate, f_SaleEffectiveDate,
        f_SaleShipmentNumber, f_PurchaseNumber, f_PurchaseLine, f_SaleNumber, f_BOLNumber,
        f_MillNetWgt, f_PurchasedWgt, f_SaleWgt, f_SaleAmount, f_Margin, f_Cost,
        f_DJJGeneratedFlag, f_EnrichedTimestampUTC
    FROM {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

# 5. FINISH METADATA LOGGING

In [0]:
merge_history = spark.sql(f"DESCRIBE HISTORY {enriched_catalog}.{DestinationSchemaName}.{DestinationTableName}").filter("operation = 'WRITE'").first()
metrics = merge_history["operationMetrics"]
RowsInserted = int(metrics.get("numTargetRowsInserted", 0))
RowsUpdated = int(metrics.get("numTargetRowsUpdated", 0))

print(f"MERGE completed - Inserted: {RowsInserted}, Updated: {RowsUpdated}")

print("ETL fact_profit_d365 completed successfully.")

MERGE completed - Inserted: 0, Updated: 0
ETL fact_profit_d365 completed successfully.


In [0]:
# ==================================
# FINISH METADATA LOGGING
# ==================================

metrics = {
    "rows_inserted": RowsInserted,
    "rows_updated": RowsUpdated,
    "rows_deleted": RowsDeleted
}

end_execution_run(run_ctx, status="SUCCESS", metrics=metrics)

In [0]:
# ==================================
# CLEANUP : DROP ALL TEMP TABLES
# ==================================

spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_fiscalmonthdates_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_grades_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_mgbushnell_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_glstaging_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_glaggregated_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_ledgerjournal_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_genkeep_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_genremove_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_purchinvline_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_salesinvline_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_factprofit_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_genkeepvend_d365")
spark.sql(f"DROP TABLE IF EXISTS {enriched_catalog}.{temp_schema}.temp_brk_genremovevend_d365")

DataFrame[]